# 14 · Modelado: Random Forest (ensamble de comparación) — mismo idioma que el gradient boosting

> **En BigQuery.** Lee `MUNICIPIO_ANIO` de `DATASET_DEFORESTACION` y, para comprobar que modela lo mismo, la muestra que guardó el gradient boosting en `DATASET_RESULTADOS`. Guarda sus resultados en `DATASET_RESULTADOS`, en tablas que empiezan por `RANDOM_FOREST_`. **Qué tienes que cambiar:** nada; la celda 1 ya apunta al proyecto del Observatorio (`prueba-ofr`). **Cuánto tarda:** unos 15 minutos.
>
> Las tablas, gráficas y frases que calcula el notebook usan los años de la celda 1. Los textos escritos a mano (como este) describen la corrida del trabajo de grado, que entrena con 2021-2024 y prueba con 2025; si cambias los años, esos textos no se actualizan.

Entrena y evalúa el bosque aleatorio sobre el panel `MUNICIPIO_ANIO`, **con la misma estructura, las mismas
variables, la misma muestra, la misma partición y las mismas métricas de `13_modelos_gradient_boosting`**, para que
las tres técnicas del equipo (gradient boosting, bosque aleatorio y regresión logística) se puedan comparar
número contra número. Las versiones anteriores de este notebook se pueden consultar en el historial de git del repositorio.

Como en el gradient boosting, el panel anual predice dos cosas sobre la misma unidad, municipio-año:

1. **¿Deforesta?** (`tasa > 0`, clasificación) — se entrena con todas las filas.
2. **Dado que deforesta, ¿cuánto?** (la tasa del IDEAM, % del bosque, regresión) — se entrena y se evalúa solo
   con los municipios-año con `tasa > 0`, porque mezclar los ceros estructurales con la magnitud distorsiona
   ambas estimaciones.

**Qué se unificó con el gradient boosting (y por qué importa).** Una comparación entre modelos solo es justa si
todos responden la misma pregunta con la misma información y se miden con la misma vara. Por eso aquí:

| Elemento | Gradient boosting (referencia) | Este notebook |
|---|---|---|
| Muestra | 3.195 municipios-año rurales, 2021-2025, con bosque | **Idéntica** (se verifica fila a fila contra el notebook del gradient boosting) |
| Variables | 15 explicativas del **mismo año**, sin rezago, como intensidades (crédito y ganado por km², cultivos y bosque como % del municipio) | **Las mismas 15**, con los mismos nombres y las mismas fórmulas |
| Partición | Entrena 2021-2024, prueba 2025 | Igual |
| Validación | `GroupKFold` por **departamento** (5 pliegues) en la búsqueda y en la prueba de territorios | Igual (la versión anterior también buscaba agrupando por municipio; aquí se conserva solo como contraste, ver 4.1) |
| Selección de hiperparámetros | Exactitud balanceada (¿deforesta?) y R² (¿cuánto?) | Igual |
| Umbral de clasificación | 0,5 | 0,5 |
| Métricas | Exactitud balanceada, AUC, precisión, recall, especificidad, F1, AP; R², RMSE, MAE, MedAE | Las mismas, más dos lecturas robustas para "¿cuánto?" (R² en escala log y Spearman) |
| Lecturas | ROC, precisión-recall, ganancia, calibración, departamentos fuera, riesgo por departamento, importancia, permutación con valor p, SHAP, lista de municipios | Las mismas, en el mismo orden de secciones |
| Hiperparámetros | Tabla 27: `learning_rate`, `max_depth`, `min_child_samples`... | Tabla 27, fila "Bosque aleatorio": `n_estimators` = 500; `max_depth` ∈ {4, 6, 8, sin límite}; `min_samples_leaf` ∈ {5, 10, 20}; `max_features` ∈ {raíz de p, 0,5} |

**Dos ampliaciones a la Tabla 27, con su razón.** La versión anterior de este notebook encontró que (a) con departamentos fuera
conviene que cada división vea todas las variables (`max_features` = 1,0) y (b) la grilla premia hojas más
grandes, que están en el borde de la tabla. Por eso la búsqueda agrega `max_features` = 1,0 y
`min_samples_leaf` = 40 (el mismo máximo de `min_child_samples` del gradient boosting), y deja que
`class_weight` (ninguno o `balanced_subsample`) lo decida la validación, no una regla fija.

**Lo que se rescató de la versión anterior** (porque aporta algo que el gradient boosting no tiene):

- El contraste entre agrupar la validación por municipio y por departamento (4.1).
- El error *out-of-bag* como segunda medida honesta de sobreajuste (5).
- La comparación de especificaciones — estructural, principal, con persistencia y con región — y la línea base de
  persistencia (5.4): responde si el crédito y la producción aportan algo más allá de "el bosque y lo que pasó el
  año pasado".
- La importancia por **bloques** y por **actividad productiva** (7.5) y las curvas de dependencia parcial (7.6),
  que evitan concluir que una actividad "no importa" solo porque sus variables, correlacionadas, se reparten el
  peso.

**Lo que se agregó pensando en los objetivos.** Al final de cada bloque hay una lectura calculada (no escrita a
mano, para que nunca contradiga a los números) vinculada a OBJ-N1/CEN-2 (priorización), ODM-2 (dónde coinciden
alto crédito y alto riesgo), ODM-3/CDM-5 (variables asociadas, con valor p corregido por Holm), CDM-6
(reproducibilidad, con una prueba explícita) y CDM-7 (brecha entrenamiento-prueba). La sección 9 las reúne.

**Alcance.** Solo el panel anual municipio × año. El panel mensual queda fuera, igual que en el gradient
boosting.

## 0. Preparación

In [ ]:
# =====================================================================
# 1. CONFIGURACIÓN
#    Ya está llena para el proyecto del Observatorio (prueba-ofr). Solo
#    hay que cambiarla si las tablas se pasan a otro proyecto o dataset.
# =====================================================================
# --- Dónde están las tablas en BigQuery ------------------------------
# Proyecto de Google Cloud del Observatorio (Prueba-OFR). Es el id,
# no el nombre: el que aparece en la consola al elegir el proyecto.
PROYECTO = "prueba-ofr"

# Dataset con las tablas de deforestación que arman los notebooks 01
# a 06 (grilla, paneles y panel municipal).
DATASET_DEFORESTACION = "Cambio_Climatico"

# Dataset donde quedan los resultados de la segmentación y de los
# modelos. Es el mismo de arriba. Para tenerlos aparte, pon otro
# nombre: el primer notebook que escriba ahí crea el dataset.
DATASET_RESULTADOS = "Cambio_Climatico"

# Ubicación de los datos. Debe ser la misma de las tablas del
# Observatorio (en BigQuery: abre una tabla > Detalles > Ubicación de
# los datos). Las del Observatorio están hoy en US.
UBICACION = "US"

# --- Años del modelo ---------------------------------------------------
# Se entrena con ANIO_INICIO a ANIO_PRUEBA - 1 y se prueba con ANIO_PRUEBA.
# Los del trabajo de grado son 2021 y 2025: entrena 2021-2024, prueba 2025.
# Para mover el modelo hacia adelante (por ejemplo, probar con 2026 cuando
# el panel municipal lo tenga completo), cambia ANIO_PRUEBA. Usa los mismos
# años en los notebooks 13 a 17.
ANIO_INICIO = 2021
ANIO_PRUEBA = 2025

# --- Solo si corres el notebook en tu computador ----------------------
# En BigQuery déjalo vacío. En un computador: la ruta al archivo .json
# de la llave de una cuenta de servicio con acceso a estas tablas.
RUTA_LLAVE_BIGQUERY = ""

In [ ]:
# =====================================================================
# 2. INSTALAR PAQUETES
#    En BigQuery instala lo que le falte a la máquina; la primera vez
#    tarda uno o dos minutos. No hay que cambiar nada. En un computador
#    con el entorno del repositorio (requirements_local.txt) se puede
#    saltar.
#    Las versiones fijas son las que se usaron en el trabajo de grado,
#    para que los resultados salgan iguales.
# =====================================================================
%pip install --quiet scikit-learn==1.6.1 statsmodels==0.14.6 shap seaborn pandas-gbq

In [ ]:
# =====================================================================
# 3. CONEXIÓN CON BIGQUERY Y FUNCIONES DE APOYO
#    No hay que cambiar nada en esta celda. Se conecta a BigQuery y
#    define las funciones con que el resto del notebook lee y escribe
#    tablas. Es igual en todos los notebooks de esta carpeta.
# =====================================================================
import json
import re
import tempfile
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from google.cloud import bigquery

NOMBRE_NOTEBOOK = "14_modelos_random_forest"

# --- 3.1 ¿Quedó algo sin llenar en la celda 1? --------------------------
# Si queda algún <<LLENAR ...>>, el notebook se detiene aquí y dice cuál.
_pendientes = sorted(n for n, v in list(globals().items())
                     if n.isupper() and isinstance(v, str) and "<<LLENAR" in v)
if _pendientes:
    raise ValueError("Falta llenar en la celda 1: " + ", ".join(_pendientes))

# --- 3.2 Conexión ---------------------------------------------------------
# En BigQuery el notebook entra con la cuenta de quien lo corre y no
# necesita llave. La llave solo se usa si se corre en un computador.
if globals().get("RUTA_LLAVE_BIGQUERY"):
    from google.oauth2 import service_account
    client = bigquery.Client(
        project=PROYECTO,
        credentials=service_account.Credentials.from_service_account_file(RUTA_LLAVE_BIGQUERY))
else:
    client = bigquery.Client(project=PROYECTO)
print(f"Conectado a BigQuery: proyecto {PROYECTO}")

# Carpeta para los archivos que se descargan mientras corre el notebook.
# Es de paso: todo lo que importa termina en tablas de BigQuery.
_carpeta = globals().get("CARPETA_DESCARGAS") or ""
DIR_TRABAJO = Path(_carpeta) if _carpeta else Path(tempfile.gettempdir()) / "ofr_deforestacion"
DIR_TRABAJO.mkdir(parents=True, exist_ok=True)


# --- 3.3 Nombres de tabla -----------------------------------------------
def nombre_completo(tabla, dataset=None):
    """'panel_ideam' -> 'proyecto.dataset.PANEL_IDEAM'.

    Las tablas que crean estos notebooks se nombran en mayúscula
    (GRILLA, PANEL_IDEAM...). Si la tabla ya viene completa
    (proyecto.dataset.tabla, como las del Observatorio), se devuelve igual.
    Si no, se arma con PROYECTO y el dataset indicado; por defecto,
    DATASET_DEFORESTACION.
    """
    if tabla.count(".") == 2:
        return tabla
    return f"{PROYECTO}.{dataset or globals().get('DATASET_DEFORESTACION')}.{tabla.upper()}"


# --- 3.4 Leer -------------------------------------------------------------
_MARCA = re.compile(r"^(\[índice(?: sin nombre)?\] )?(?:Nombre original: «(.*?)»)?")


_LECTURA_RAPIDA = [True]


def leer_sql(sql):
    """Corre una consulta SQL en BigQuery y devuelve un DataFrame de pandas.

    Usa la lectura rápida de BigQuery (Storage Read API). Si no está
    disponible (por ejemplo, porque la cuenta no tiene el rol «BigQuery
    Read Session User»), sigue con la lectura normal, que da el mismo
    resultado pero tarda más en tablas grandes.
    """
    trabajo = client.query(sql)
    trabajo.result()                      # si la consulta tiene un error, sale aquí
    resultado = None
    if _LECTURA_RAPIDA[0]:
        try:
            resultado = trabajo.to_arrow()
        except Exception as e:
            _LECTURA_RAPIDA[0] = False
            print("Aviso: la lectura rápida de BigQuery no está disponible "
                  f"({type(e).__name__}); se usa la lectura normal.")
    if resultado is None:
        resultado = trabajo.to_arrow(create_bqstorage_client=False)
    return resultado.to_pandas(date_as_object=False, coerce_temporal_nanoseconds=True)


def leer_tabla(tabla, columnas=None, dataset=None, filtro=None):
    f"""Lee una tabla de BigQuery a pandas.

    columnas  lista de columnas a traer; por defecto, todas.
    filtro    condición SQL opcional, p. ej. "anio >= {ANIO_INICIO}".

    Si la tabla se guardó con guardar_tabla(), devuelve las filas en el
    mismo orden en que se guardaron, las columnas con su nombre original
    (tildes, espacios) y el índice.
    """
    destino = nombre_completo(tabla, dataset)
    esquema = client.get_table(destino).schema
    con_orden = any(c.name == "orden_fila" for c in esquema)
    seleccion = ", ".join(f"`{c}`" for c in columnas) if columnas else "*"
    df = leer_sql(f"SELECT {seleccion} FROM `{destino}`"
                  + (f" WHERE {filtro}" if filtro else "")
                  + (" ORDER BY orden_fila" if con_orden else ""))
    df = df.drop(columns=["orden_fila"], errors="ignore")

    originales, indice, sin_nombre = {}, [], set()
    for campo in esquema:
        marca = _MARCA.match(campo.description or "")
        if marca.group(2):
            originales[campo.name] = marca.group(2)
        if marca.group(1) and campo.name in df.columns:
            indice.append(campo.name)
            if "sin nombre" in marca.group(1):
                sin_nombre.add(campo.name)
    if indice:
        df = df.set_index(indice)
        df.index = df.index.set_names(
            [None if n in sin_nombre else originales.get(n, n) for n in indice])
    return df.rename(columns=originales)


# --- 3.5 Escribir -----------------------------------------------------------
_DATASETS_LISTOS = set()


def _nombre_para_bigquery(nombre, usados):
    """Nombre de columna que BigQuery acepta siempre: letras sin tilde,
    números y guion bajo. Los que ya cumplen se dejan igual."""
    texto = str(nombre)
    if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,299}", texto) and texto.lower() not in usados:
        usados.add(texto.lower())
        return texto
    limpio = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    limpio = re.sub(r"[^0-9A-Za-z]+", "_", limpio).strip("_").lower() or "columna"
    if limpio[0].isdigit():
        limpio = "c_" + limpio
    limpio, base, k = limpio[:280], limpio[:280], 2
    while limpio.lower() in usados:
        limpio, k = f"{base}_{k}", k + 1
    usados.add(limpio.lower())
    return limpio


def _tipos_para_bigquery(df):
    """Deja cada columna en un tipo que BigQuery reciba sin error:
    categorías a texto, enteros con vacíos a entero, mezclas a texto."""
    for c in df.columns:
        s = df[c]
        if isinstance(s.dtype, pd.CategoricalDtype):
            s = s.astype(object).where(s.notna(), None)
        if s.dtype == object:
            valores = s.dropna()
            tipos = {type(v) for v in valores}
            if not tipos or tipos <= {str}:
                pass
            elif all(isinstance(v, (bool, np.bool_)) for v in valores):
                s = s.astype("boolean")
            elif all(isinstance(v, (int, np.integer)) and not isinstance(v, bool) for v in valores):
                s = s.astype("Int64")
            elif all(isinstance(v, (int, float, np.integer, np.floating)) for v in valores):
                s = s.astype(float)
            else:
                s = s.map(lambda v: None if v is None or v is pd.NA
                          or (isinstance(v, float) and v != v) else str(v))
        df[c] = s
    return df


def guardar_tabla(df, tabla, descripcion, dataset=None, indice=False,
                  descripciones=None, procedencia=None):
    """Escribe `df` en BigQuery. Si la tabla ya existe, la REEMPLAZA entera.

    tabla          nombre corto, p. ej. "panel_ideam".
    descripcion    texto que se ve en BigQuery, en los detalles de la tabla.
    dataset        por defecto, DATASET_DEFORESTACION. Si no existe, se crea.
    indice         True para guardar también el índice de pandas como
                   columna(s); leer_tabla() lo recupera.
    descripciones  {columna: texto}, para documentar columnas en BigQuery.
    procedencia    {"fuente": ..., ...}: si se da, deja una fila en la tabla
                   registro_cargas con qué se cargó, de dónde y cuándo.
    """
    dataset = dataset or globals().get("DATASET_DEFORESTACION")
    destino = nombre_completo(tabla, dataset)
    if dataset not in _DATASETS_LISTOS:
        ds = bigquery.Dataset(f"{PROYECTO}.{dataset}")
        ds.location = globals().get("UBICACION", "US")
        client.create_dataset(ds, exists_ok=True)
        _DATASETS_LISTOS.add(dataset)

    datos = df.copy()
    marcas = {}
    if indice:
        nombres_indice = list(datos.index.names)
        datos = datos.reset_index()
        for k, n in enumerate(nombres_indice):
            columna = datos.columns[k]
            marcas[columna] = "[índice sin nombre] " if n is None else "[índice] "
    else:
        datos = datos.reset_index(drop=True)
    if isinstance(datos.columns, pd.MultiIndex):
        datos.columns = [" | ".join(str(x) for x in c if str(x) != "") for c in datos.columns]

    descripciones = descripciones or {}
    usados, notas, columnas = set(), {}, []
    for c in datos.columns:
        nuevo = _nombre_para_bigquery(c, usados)
        texto = marcas.get(c, "")
        if nuevo != str(c):
            texto += f"Nombre original: «{c}»"
        if descripciones.get(c):
            texto += (". " if texto and not texto.endswith("] ") else "") + descripciones[c]
        notas[nuevo] = texto[:1024]
        columnas.append(nuevo)
    datos.columns = columnas
    datos = _tipos_para_bigquery(datos)
    # BigQuery no guarda el orden de las filas, y algunos cálculos (el
    # muestreo del random forest, por ejemplo) dependen de él. Esta
    # columna lo conserva; leer_tabla() la usa y la quita.
    datos["orden_fila"] = np.arange(len(datos), dtype="int64")
    notas["orden_fila"] = ("Posición de la fila en la tabla original. Sirve para leerla "
                           "en el mismo orden: ORDER BY orden_fila.")

    trabajo = client.load_table_from_dataframe(
        datos, destino,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    trabajo.result()

    t = client.get_table(destino)
    t.description = descripcion
    t.schema = [bigquery.SchemaField(f.name, f.field_type, mode=f.mode,
                                     description=notas.get(f.name) or None,
                                     fields=f.fields) for f in t.schema]
    client.update_table(t, ["description", "schema"])
    print(f"Guardada {destino}: {t.num_rows:,} filas, {len(t.schema)} columnas")

    if procedencia is not None:
        fila = pd.DataFrame([{
            "tabla": tabla.upper(),
            "filas": int(t.num_rows),
            "fuente": str(procedencia.get("fuente", "")),
            "parametros": json.dumps({k: v for k, v in procedencia.items() if k != "fuente"},
                                     ensure_ascii=False, default=str),
            "notebook": NOMBRE_NOTEBOOK,
            "cargado_en": datetime.now(timezone.utc),
        }])
        client.load_table_from_dataframe(
            fila, nombre_completo("registro_cargas", dataset),
            job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")).result()

In [ ]:
import json
import os
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from joblib import Parallel, delayed
from scipy import stats
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (average_precision_score, balanced_accuracy_score, confusion_matrix,
                              f1_score, mean_absolute_error, mean_squared_error, median_absolute_error,
                              precision_recall_curve, precision_score, r2_score, recall_score,
                              roc_auc_score, roc_curve)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from statsmodels.stats.multitest import multipletests

sns.set_theme(style="whitegrid", font_scale=1.1, palette="colorblind")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 13})
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)


def guardar_resultado(df, nombre, indice=False, descripcion=None):
    """Guarda un resultado en DATASET_RESULTADOS. En el repositorio, cada uno era un CSV de
    "modelos y clusters/resultados/" con este mismo nombre."""
    if isinstance(df, pd.Series):
        df = df.to_frame()
    guardar_tabla(df, nombre, descripcion or f"Resultado del notebook {NOMBRE_NOTEBOOK}: {nombre.replace('_', ' ')}.",
                  dataset=DATASET_RESULTADOS, indice=indice)
    return df


SEMILLA = 0   # la misma del gradient boosting (CDM-6: reproducibilidad)
UMBRAL = 0.5  # el que usa .predict() del gradient boosting: todos los modelos clasifican con el mismo umbral

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import partial_dependence
from sklearn.preprocessing import OneHotEncoder

## 1. Cargar el panel y aplicar los filtros de 4.1

Mismos filtros que `09_eda_panel_municipal` y que el gradient boosting: municipios rurales o rurales
dispersos, según la clasificación del DNP que trae la tabla de municipios del Observatorio.

In [ ]:
RURALES = ["Rural", "Rural disperso"]

anual = leer_tabla("municipio_anio")
anual = anual[anual.ruralidad.isin(RURALES)].copy()

print(f"Filas: {len(anual):,}  ·  Municipios rurales: {anual.cod_dane.nunique():,}  ·  "
      f"Años: {sorted(int(a) for a in anual.anio.unique())}")
anual[["cod_dane", "anio", "nombre_municipio", "nombre_departamento", "ruralidad"]].head()

## 2. Las 16 variables del dataset final (15 explicativas + objetivo)

Se construyen **exactamente** como en el gradient boosting: crédito de producción (total y por cadena) y ganado
por km², cultivos como % del área del municipio (área **cosechada** de la EVA), y los dos atributos fijos del
municipio, bosque y área. Todas del **mismo año** que la tasa (sin rezago).

> **Un cambio respecto de la versión anterior.** Esa versión usaba montos en log(1+x) del año anterior (*t − 1*), con
> 15 variables distintas (incluidas región y ruralidad). Eso hacía que el bosque aleatorio y el gradient
> boosting respondieran preguntas ligeramente distintas — uno "anticipaba", el otro describía el mismo año —, y
> que sus resultados no fueran comparables. Aquí se alinea con el gradient boosting, que fue la decisión del
> interesado. La versión con rezago y la región se conservan como sensibilidad en 5.4, para que no se pierda
> lo aprendido.

Para un bosque aleatorio las transformaciones monótonas (logaritmo, estandarización) no cambian las particiones,
así que **no se transforma nada**: los árboles reciben las variables tal cual, con los ceros y los valores
extremos que el negocio busca (OBJ-N1: "los atípicos no se borran").

In [ ]:
km2 = anual.area_grilla_ha / 100
panel = anual.assign(
    tasa=100 * anual.tasa_def_ideam,
    cred=anual.credito_produccion_mm / km2,
    cred_gan=anual.credito_ganaderia_produccion_mm / km2,
    cred_palma=anual.credito_palma_produccion_mm / km2,
    cred_cacao=anual.credito_cacao_produccion_mm / km2,
    cred_cafe=anual.credito_cafe_produccion_mm / km2,
    cred_arroz=anual.credito_arroz_produccion_mm / km2,
    bovinos=anual.bovinos / km2,
    fincas=100 * anual.fincas_bovinas / km2,
    agricola=100 * anual.area_cosechada_ha / anual.area_grilla_ha,
    palma=100 * anual.cosechada_palma_ha / anual.area_grilla_ha,
    cacao=100 * anual.cosechada_cacao_ha / anual.area_grilla_ha,
    cafe=100 * anual.cosechada_cafe_ha / anual.area_grilla_ha,
    arroz=100 * anual.cosechada_arroz_ha / anual.area_grilla_ha,
    bosque=100 * anual.bosque_ideam_inicial_ha / anual.area_grilla_ha,
    area=km2,
).replace([np.inf, -np.inf], np.nan)

# Las 15 explicativas del dataset final (iguales a VARIABLES en 09_eda_panel_municipal y a
# EXPLICATIVAS en 13_modelos_gradient_boosting)
EXPLICATIVAS = ["cred", "cred_gan", "cred_palma", "cred_cacao", "cred_cafe", "cred_arroz", "bovinos",
                 "fincas", "agricola", "palma", "cacao", "cafe", "arroz", "bosque", "area"]

NOMBRES = {
    "cred": "Crédito de producción (M$/km²)", "cred_gan": "Crédito ganadero de producción (M$/km²)",
    "cred_palma": "Crédito de producción a palma (M$/km²)", "cred_cacao": "Crédito de producción a cacao (M$/km²)",
    "cred_cafe": "Crédito de producción a café (M$/km²)", "cred_arroz": "Crédito de producción a arroz (M$/km²)",
    "bovinos": "Bovinos por km²", "fincas": "Fincas ganaderas por 100 km²",
    "agricola": "Área cosechada (% del municipio)", "palma": "Palma (% del municipio)",
    "cacao": "Cacao (% del municipio)", "cafe": "Café (% del municipio)", "arroz": "Arroz (% del municipio)",
    "bosque": "Bosque (% del municipio)", "area": "Área del municipio (km²)",
}

columnas = ["cod_dane", "nombre_municipio", "anio", "nombre_departamento", "tasa", "def_ideam_ha"] + EXPLICATIVAS
panel = panel[columnas].sort_values(["cod_dane", "anio"]).copy()

# La EVA no tiene 2025: se completa con el ultimo cultivo conocido de cada municipio (2024) en vez de
# perder todas las filas de 2025 al hacer dropna (misma decision que el gradient boosting).
CULTIVOS = ["agricola", "palma", "cacao", "cafe", "arroz"]
panel[CULTIVOS] = panel.groupby("cod_dane")[CULTIVOS].ffill()

# El objetivo solo se evalua 2021-2025 (lo que tiene cifra del IDEAM); donde no hay bosque la tasa
# queda vacia (3.4) y se descarta.
muestra = panel[panel.anio.between(ANIO_INICIO, ANIO_PRUEBA) & panel.tasa.notna()].dropna(subset=EXPLICATIVAS).copy()
muestra["deforesta"] = (muestra.tasa > 0).astype(int)

print(f"Filas utilizables ({ANIO_INICIO}-{ANIO_PRUEBA}, con bosque): {len(muestra):,}")
print(f"Municipios-año con deforestación > 0: {muestra.deforesta.sum():,} "
      f"({100 * muestra.deforesta.mean():.1f} %)")
muestra[["cod_dane", "anio", "deforesta", "tasa"] + EXPLICATIVAS].head()

In [ ]:
# Verificación de "mismo idioma": la muestra tiene que ser la misma del gradient boosting (notebook 13), que
# la guardó en BigQuery: mismas filas y mismos valores en las 15 variables.
try:
    a = leer_tabla("gradient_boosting_muestra", dataset=DATASET_RESULTADOS)
except Exception:
    a = None
    print("Aviso: no está la tabla gradient_boosting_muestra (se crea al correr el notebook 13); "
          "no se pudo verificar que la muestra sea la misma.")
if a is not None:
    a = a.sort_values(["cod_dane", "anio"]).reset_index(drop=True)
    b = muestra.sort_values(["cod_dane", "anio"]).reset_index(drop=True)
    assert len(a) == len(b) and (a.cod_dane.astype(str) == b.cod_dane.astype(str)).all() and (a.anio == b.anio).all(), \
        "las filas no coinciden con las del gradient boosting: ¿usaste los mismos años y el mismo panel en el notebook 13?"
    assert np.allclose(a[EXPLICATIVAS].values.astype(float), b[EXPLICATIVAS].values.astype(float), equal_nan=True), \
        "los valores de las 15 variables no coinciden con los del gradient boosting"
    print(f"OK: mismas {len(b):,} filas y mismos valores en las 15 variables que el gradient boosting (notebook 13).")

# A partir de aqui el departamento es texto (evita categorias vacias al agrupar)
muestra["nombre_departamento"] = muestra.nombre_departamento.astype(str)

## 3. Partición: 2021-2024 para entrenar, 2025 para probar

Igual que el gradient boosting: se entrena con 2021 a 2024 y se prueba con 2025, sin tocar 2025 durante la
búsqueda de hiperparámetros. El departamento se guarda aparte para la validación por departamentos completos,
tanto dentro de la búsqueda (sección 4) como en la evaluación final independiente del corte temporal (sección 6).

In [ ]:
train = muestra[muestra.anio.between(ANIO_INICIO, ANIO_PRUEBA - 1)].reset_index(drop=True)
test = muestra[muestra.anio == ANIO_PRUEBA].reset_index(drop=True)

X_train, y_train_clas, y_train_tasa = train[EXPLICATIVAS], train.deforesta, train.tasa
X_test, y_test_clas, y_test_tasa = test[EXPLICATIVAS], test.deforesta, test.tasa
grupos_train = train.nombre_departamento

# Mismos pliegues que el gradient boosting: GroupKFold por departamento, 5 pliegues, es determinista
N_PLIEGUES = 5

print(f"Entrena ({ANIO_INICIO}-{ANIO_PRUEBA - 1}): {len(train):,} filas, {grupos_train.nunique()} departamentos")
print(f"Prueba ({ANIO_PRUEBA}): {len(test):,} filas")
pd.DataFrame({"filas": [len(train), len(test)], "no deforesta": [(y_train_clas == 0).sum(), (y_test_clas == 0).sum()],
              "deforesta": [(y_train_clas == 1).sum(), (y_test_clas == 1).sum()],
              "% deforesta": [100 * y_train_clas.mean(), 100 * y_test_clas.mean()]},
             index=[f"entrenamiento {ANIO_INICIO}-{ANIO_PRUEBA - 1}", f"prueba {ANIO_PRUEBA}"]).round(1)

## 4. Búsqueda de hiperparámetros (Tabla 27)

Es la sesión de ajuste: encontrar la combinación que mejor generaliza antes de evaluar nada en 2025. La técnica es una **grilla exhaustiva** (*grid search*): la grilla
completa se recorre (no se muestrea ni se usa optimización bayesiana): `max_depth` ∈ {4, 6, 8, sin límite} × `min_samples_leaf` ∈ {5, 10, 20, 40} ×
`max_features` ∈ {raíz de p, 0,5, 1,0} × `class_weight` ∈ {ninguno, `balanced_subsample`} = 96 combinaciones para
"¿deforesta?", y 48 para "¿cuánto?" (que no usa pesos de clase). Con 500 árboles cada una.

**Por qué cada hiperparámetro importa aquí.**

- **`max_depth` y `min_samples_leaf`.** Fijan cuánto puede "memorizar" cada árbol. Con solo 639 municipios y unos
  pocos extremos reales (Cartagena del Chairá, San Vicente del Caguán, Pedraza en la tasa), una hoja construida
  sobre uno o dos municipios es una regla a la medida de esos casos. Hojas más grandes obligan a que cada regla
  se sostenga en varios municipios, que es la forma de tratar los atípicos sin borrarlos y de vigilar la brecha
  entrenamiento-prueba que fija CDM-7.
- **`max_features`.** Cuántas variables mira cada división. Menos variables decorrelaciona los árboles (más
  estabilidad); todas las variables deja que el bosque —la variable dominante— compita en cada división. Con
  departamentos fuera La versión anterior de este notebook encontró que esto último conviene.
- **`class_weight`.** Compensa que haya más municipios-año que deforestan (63 %) que los que no. Puede mejorar la
  exactitud balanceada, pero **distorsiona la probabilidad** (la calibración), que importa cuando el
  Observatorio la usa como número y no solo como orden. Por eso se deja que la validación decida.

**Cómo se valida cada combinación.** `GroupKFold` por departamento (5 pliegues) sobre 2021-2024, con los mismos
pliegues del gradient boosting; **2025 no se toca en esta sección**. La métrica de selección es la misma: exactitud
balanceada al umbral de 0,5 para "¿deforesta?" y R² para "¿cuánto?". Se registran además el AUC y la desviación
entre pliegues, que dicen qué tan estable es la elección.

In [ ]:
from itertools import product

FIJOS_RF = dict(n_estimators=500, random_state=SEMILLA)  # Tabla 27: 500 arboles, estabilidad del promedio

GRID_CLAS = [dict(max_depth=d, min_samples_leaf=l, max_features=f, class_weight=w)
             for d, l, f, w in product((4, 6, 8, None), (5, 10, 20, 40), ("sqrt", 0.5, 1.0), (None, "balanced_subsample"))]
GRID_REG = [dict(max_depth=d, min_samples_leaf=l, max_features=f)
            for d, l, f in product((4, 6, 8, None), (5, 10, 20, 40), ("sqrt", 0.5, 1.0))]


def _pliegue(modelo_cls, params, X, y, i_train, i_val, tipo):
    modelo = modelo_cls(**FIJOS_RF, **params, n_jobs=1).fit(X.iloc[i_train], y.iloc[i_train])
    yv = y.iloc[i_val]
    if tipo == "clas":
        prob = modelo.predict_proba(X.iloc[i_val])[:, 1]
        return dict(metrica=balanced_accuracy_score(yv, prob >= UMBRAL), auc=roc_auc_score(yv, prob),
                    ap=average_precision_score(yv, prob))
    pred = modelo.predict(X.iloc[i_val])
    return dict(metrica=r2_score(yv, pred), mae=mean_absolute_error(yv, pred), r2_log=r2_score(np.log(yv), np.log(np.clip(pred, 1e-4, None))))


def buscar(modelo_cls, X, y, grupos, grid, tipo):
    # Recorre `grid`, evalua cada combinacion con GroupKFold por departamento (los mismos pliegues del
    # gradient boosting) y devuelve (id de la mejor combinacion, tabla de resultados). Los ajustes por
    # (combinacion, pliegue) corren en paralelo; cada bosque usa una semilla fija, asi que el resultado no
    # depende de cuantos nucleos tenga la maquina (CDM-6).
    pliegues = list(GroupKFold(n_splits=N_PLIEGUES).split(X, y, groups=grupos))
    tareas = [(k, j) for k in range(len(grid)) for j in range(len(pliegues))]
    inicio = time.time()
    salidas = Parallel(n_jobs=-1)(delayed(_pliegue)(modelo_cls, grid[k], X, y, *pliegues[j], tipo) for k, j in tareas)
    filas = []
    for k, params in enumerate(grid):
        rs = pd.DataFrame([s for (kk, _), s in zip(tareas, salidas) if kk == k])
        filas.append({"id": k, **{h: (str(v) if h in ("max_depth", "class_weight") else v) for h, v in params.items()}, **rs.mean().to_dict(),
                      "metrica_sd": rs.metrica.std()})
    tabla = pd.DataFrame(filas).sort_values("metrica", ascending=False, kind="stable").reset_index(drop=True)
    print(f"{len(grid)} combinaciones x {len(pliegues)} pliegues en {(time.time() - inicio) / 60:.1f} min")
    return int(tabla.loc[0, "id"]), tabla


def describir(params):
    return ", ".join(f"{k}={v}" for k, v in params.items())

In [ ]:
# Clasificacion: ¿deforesta? -> se maximiza la exactitud balanceada (hay mas unos que ceros)
id_clas, tabla_clas = buscar(RandomForestClassifier, X_train, y_train_clas, grupos_train, GRID_CLAS, "clas")
params_clas = GRID_CLAS[id_clas]
print("Mejor combinación (clasificación):", describir(params_clas))
tabla_clas.head(8).round(4)

In [ ]:
# Regresion: dado que deforesta, ¿cuanto? -> solo sobre las filas con deforesta == 1, se maximiza R2
en_deforesta = (y_train_clas == 1).values
id_reg, tabla_reg = buscar(RandomForestRegressor, X_train[en_deforesta], y_train_tasa[en_deforesta],
                           grupos_train[en_deforesta], GRID_REG, "reg")
params_reg = GRID_REG[id_reg]
print(f"Filas de entrenamiento con deforesta == 1: {en_deforesta.sum():,} de {len(X_train):,}")
print("Mejor combinación (regresión):", describir(params_reg))
tabla_reg.head(8).round(4)

### 4.1 Cómo se ve la búsqueda, y qué cambia al agrupar por departamento

El primer mapa muestra la exactitud balanceada de validación y el segundo, la brecha entre el AUC de
entrenamiento y el de validación (sobreajuste) para el `max_features` y el `class_weight` ganadores. Es la
forma más directa de ver si una hoja más grande o un árbol menos profundo cambian algo: si la superficie es
plana, la elección importa poco y el modelo es estable frente a los hiperparámetros.

Debajo, la combinación ganadora se evalúa también agrupando por **municipio** (la validación de la versión
original). La diferencia es el costo de ser honestos con la autocorrelación espacial: agrupar por municipio deja
que el modelo vea a los vecinos del municipio que valida, y eso infla el puntaje.

In [ ]:
mejor_fila = tabla_clas.loc[0]
mismo = tabla_clas[(tabla_clas.max_features.astype(str) == str(params_clas["max_features"]))
                   & (tabla_clas.class_weight.astype(str) == str(params_clas["class_weight"]))].copy()
mismo["max_depth"] = mismo.max_depth.astype(str)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
tabla = mismo.pivot_table(index="max_depth", columns="min_samples_leaf", values="metrica").reindex(["4", "6", "8", "None"])
sns.heatmap(tabla, annot=True, fmt=".3f", cmap="viridis", ax=ax[0], cbar=False)
ax[0].set(title="Exactitud balanceada en validación (departamentos fuera)", xlabel="min_samples_leaf", ylabel="max_depth")
tabla = mismo.pivot_table(index="max_depth", columns="min_samples_leaf", values="auc").reindex(["4", "6", "8", "None"])
sns.heatmap(tabla, annot=True, fmt=".3f", cmap="magma_r", ax=ax[1], cbar=False)
ax[1].set(title="AUC en validación (departamentos fuera)", xlabel="min_samples_leaf", ylabel="max_depth")
plt.tight_layout()
plt.show()

# Contraste: la misma combinacion ganadora, validada agrupando por municipio en vez de por departamento
def validar_agrupando_por(grupos):
    pliegues = GroupKFold(n_splits=N_PLIEGUES).split(X_train, y_train_clas, groups=grupos)
    rs = pd.DataFrame(Parallel(n_jobs=-1)(delayed(_pliegue)(RandomForestClassifier, params_clas, X_train, y_train_clas, i, v, "clas")
                                          for i, v in pliegues))
    return rs.metrica.mean(), rs.auc.mean(), rs.auc.std()

ba_dep, auc_dep, sd_dep = mejor_fila.metrica, mejor_fila.auc, np.nan
ba_mun, auc_mun, sd_mun = validar_agrupando_por(train.cod_dane)
print(f"Misma combinación ganadora, validada de dos maneras ({ANIO_INICIO}-{ANIO_PRUEBA - 1}):")
print(f"  agrupando por DEPARTAMENTO: exactitud balanceada {ba_dep:.3f}  ·  AUC {auc_dep:.3f}")
print(f"  agrupando por MUNICIPIO:    exactitud balanceada {ba_mun:.3f}  ·  AUC {auc_mun:.3f}  (desv. entre pliegues {sd_mun:.3f})")
print(f"Costo de agrupar por departamento: {auc_mun - auc_dep:+.3f} de AUC — es la parte del puntaje que venía de "
      f"validar municipios cuyos vecinos de departamento sí estaban en el ajuste.")

**Lectura: por qué la elección de hiperparámetros es defendible (CDM-7).**

- **Hay un casi empate.** Las ocho mejores combinaciones de ¿deforesta? quedan entre 0,8001 y 0,8034 de exactitud balanceada, y la desviación entre pliegues (0,045 a 0,065) es mucho mayor que la diferencia entre
  la primera y la tercera (0,8034 frente a 0,8026). La ganadora (sin límite de profundidad, hoja mínima de 10, todas las variables, `balanced_subsample`) no es "la única buena": se eligió por la regla común, no porque
  las demás sean peores en la práctica. Seis de las ocho usan `balanced_subsample`, que es lo que corrige que el 63 % de los municipios-año deforeste.
- **Qué protege contra el sobreajuste.** Una hoja mínima de 10 municipios evita reglas a la medida de uno o dos casos extremos (Pedraza, Cartagena del Chairá), que es como se tratan los atípicos sin borrarlos.
- **El costo de ser honestos.** Agrupar por departamento en vez de por municipio resta 0,013 de AUC (0,898 frente a 0,911): ese es el puntaje que venía de validar municipios cuyos vecinos ya estaban en el ajuste.
  Es la cifra que respalda que lo que se reporta después es el desempeño sobre territorios que el modelo no vio.

## 5. Modelo final y evaluación en 2025

Se reentrena cada modelo sobre todo 2021-2024 con la combinación ganadora y se evalúa una sola vez sobre 2025,
que no se tocó antes. Además de las métricas de la matriz de confusión (precisión, recall/sensibilidad,
especificidad, F1) y del AUC, se mide la brecha entrenamiento-prueba contra el 15 % de CDM-7.

**Una precaución propia del bosque aleatorio.** Un bosque casi siempre acierta mucho sobre las filas con las que
se entrenó (cada árbol vio un 63 % de ellas), así que la exactitud *dentro de la muestra* es un termómetro
optimista y la brecha medida contra ella tiende a salir grande aunque el modelo generalice bien. Por eso se
reportan tres brechas: contra el entrenamiento dentro de la muestra (la forma en que el gradient boosting
reporta la suya, para poder compararlas), contra el error *out-of-bag* (cada fila predicha solo por los
árboles que no la vieron: es una medida honesta que el bosque trae gratis) y contra la validación por
departamentos de la búsqueda (la definición literal de CDM-7: "validación cruzada con bloques geográficos").
El OOB es algo optimista aquí, porque las filas que quedan fuera de un árbol pueden ser otros años del mismo
municipio.

In [ ]:
def metricas_clasificacion(y, pred, prob, etiqueta):
    # Las mismas del gradient boosting: precision, recall/sensibilidad, especificidad, F1, AUC y AP.
    precision = precision_score(y, pred, zero_division=0)
    recall = recall_score(y, pred)
    especificidad = recall_score(y, pred, pos_label=0)
    f1 = f1_score(y, pred)
    auc = roc_auc_score(y, prob)
    ap = average_precision_score(y, prob)
    print(f"[{etiqueta}] precision: {precision:.3f}  ·  recall/sensibilidad: {recall:.3f}  ·  "
          f"especificidad: {especificidad:.3f}  ·  F1: {f1:.3f}  ·  AUC: {auc:.3f}  ·  "
          f"precisión promedio (AP): {ap:.3f}")
    return dict(precision=precision, recall=recall, especificidad=especificidad, f1=f1, auc=auc, ap=ap)


def metricas_regresion(y, pred, etiqueta):
    # R², RMSE, MAE y MedAE como en el gradient boosting. Se agregan dos lecturas que no dependen de unos
    # pocos municipios extremos (5.3 del gradient boosting): el R² en escala logaritmica y la correlacion
    # de rangos de Spearman entre la tasa real y la predicha. Se calculan solo a partir de las
    # predicciones, asi que se pueden aplicar igual a cualquier modelo (incluido el gradient boosting).
    y, pred = np.asarray(y, dtype=float), np.asarray(pred, dtype=float)
    rmse = mean_squared_error(y, pred) ** 0.5
    mae = mean_absolute_error(y, pred)
    medae = median_absolute_error(y, pred)
    r2 = r2_score(y, pred)
    r2_log = r2_score(np.log(y), np.log(np.clip(pred, 1e-4, None)))
    rho = stats.spearmanr(y, pred).statistic
    print(f"[{etiqueta}] R²: {r2:.3f}  ·  RMSE: {rmse:.3f}  ·  MAE: {mae:.3f}  ·  MedAE: {medae:.3f}  ·  "
          f"R² en escala log: {r2_log:.3f}  ·  Spearman: {rho:.3f}")
    return dict(r2=r2, rmse=rmse, mae=mae, medae=medae, r2_log=r2_log, spearman=rho)


def ver_brecha(nombre, entrenamiento, prueba, umbral=15):
    # CDM-7: brecha relativa entre el desempeño de entrenamiento y el de prueba (<= 15 %)
    brecha = 100 * (entrenamiento - prueba) / entrenamiento
    estado = "OK" if abs(brecha) <= umbral else f"por encima del {umbral} % de CDM-7"
    print(f"{nombre}: entrena {entrenamiento:.3f}  ·  prueba {prueba:.3f}  ·  brecha {brecha:.1f} % ({estado})")
    return brecha


def lecturas_prioridad(y, prob, ha, titulo, sufijo, etiqueta_modelo, color="#2a78d6"):
    # Las cuatro lecturas de la misma probabilidad que usa el gradient boosting (5.1 y 6.1): ROC,
    # precision-recall, ganancia (OBJ-N1) y calibracion. Devuelve la cobertura al revisar 10, 20 y 30 %.
    y, prob, ha = np.asarray(y), np.asarray(prob), np.asarray(ha, dtype=float)
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))

    fpr, tpr, _ = roc_curve(y, prob)
    auc = roc_auc_score(y, prob)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f"{etiqueta_modelo} (AUC = {auc:.3f})")
    axes[0].plot([0, 1], [0, 1], color="grey", ls="--", lw=1, label="Al azar")
    axes[0].set(xlabel="Tasa de falsos positivos", ylabel="Tasa de verdaderos positivos", title=f"Curva ROC\n{titulo}")
    axes[0].legend(loc="lower right", fontsize=9)

    prec_c, rec_c, _ = precision_recall_curve(y, prob)
    ap = average_precision_score(y, prob)
    axes[1].plot(rec_c, prec_c, color=color, lw=2, label=f"{etiqueta_modelo} (AP = {ap:.3f})")
    axes[1].axhline(y.mean(), color="grey", ls="--", lw=1, label=f"Al azar (= {y.mean():.2f})")
    axes[1].set(xlabel="Recall (sensibilidad)", ylabel="Precisión", title=f"Curva precisión-recall\n{titulo}")
    axes[1].legend(loc="lower left", fontsize=9)

    orden = np.argsort(-prob)
    y_ord, ha_ord = y[orden], ha[orden]
    cubierto = np.cumsum(y_ord) / y_ord.sum()
    ha_cubierta = np.cumsum(ha_ord) / ha_ord.sum()
    fraccion = np.arange(1, len(y_ord) + 1) / len(y_ord)
    axes[2].plot(100 * fraccion, 100 * cubierto, color=color, lw=2, label="% de municipios-año que sí deforestaron")
    axes[2].plot(100 * fraccion, 100 * ha_cubierta, color="#2e9e5b", lw=2, ls="-.", label="% de hectáreas realmente deforestadas")
    axes[2].plot([0, 100], [0, 100], color="grey", ls="--", lw=1, label="Priorizar al azar")
    resultado = {}
    for corte in (0.1, 0.2, 0.3):
        n = int(corte * len(y_ord))
        resultado[int(100 * corte)] = dict(n=n, pct_municipios=100 * cubierto[n - 1], pct_ha=100 * ha_cubierta[n - 1],
                                           ha=ha_ord[:n].sum(), ha_total=ha_ord.sum())
        axes[2].annotate(f"{100 * cubierto[n - 1]:.0f} %", (100 * corte, 100 * cubierto[n - 1]),
                          textcoords="offset points", xytext=(5, -10), fontsize=9)
    axes[2].set(xlabel="% de municipios-año revisados", ylabel="% cubierto", title=f"Curva de ganancia (OBJ-N1)\n{titulo}")
    axes[2].legend(loc="lower right", fontsize=8)

    frac_pos, prob_media = calibration_curve(y, prob, n_bins=8, strategy="quantile")
    axes[3].plot(prob_media, frac_pos, "o-", color=color, lw=2, label=etiqueta_modelo)
    axes[3].plot([0, 1], [0, 1], color="grey", ls="--", lw=1, label="Calibración perfecta")
    axes[3].set(xlabel="Probabilidad predicha (promedio del grupo)", ylabel="Fracción que sí deforestó",
                title=f"Curva de calibración\n{titulo}")
    axes[3].legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.show()
    for corte, r in resultado.items():
        print(f"Revisando el {corte} % con mayor riesgo predicho ({r['n']} municipios-año, {sufijo}) se cubre el "
              f"{r['pct_municipios']:.0f} % de los que sí deforestaron y el {r['pct_ha']:.0f} % de las hectáreas "
              f"realmente deforestadas ({r['ha']:,.0f} de {r['ha_total']:,.0f} ha).")
    return resultado


def matriz_confusion(y, pred, titulo):
    matriz = confusion_matrix(y, pred)
    fig, ax = plt.subplots(figsize=(5, 4.5))
    sns.heatmap(matriz, annot=True, fmt=",d", cmap="Blues", cbar=False,
                xticklabels=["No deforesta", "Deforesta"], yticklabels=["No deforesta", "Deforesta"], ax=ax)
    ax.set(xlabel="Predicho", ylabel="Real", title=titulo)
    plt.show()
    return matriz

In [ ]:
clasificador = RandomForestClassifier(**FIJOS_RF, **params_clas, oob_score=True, n_jobs=-1)
clasificador.fit(X_train, y_train_clas)

pred_train = clasificador.predict(X_train)
pred_test = clasificador.predict(X_test)
prob_test = clasificador.predict_proba(X_test)[:, 1]
prob_oob = clasificador.oob_decision_function_[:, 1]

ba_train, ba_test = balanced_accuracy_score(y_train_clas, pred_train), balanced_accuracy_score(y_test_clas, pred_test)
ba_oob = balanced_accuracy_score(y_train_clas, prob_oob >= UMBRAL)
ba_cv_dep = mejor_fila.metrica
brecha_clas = ver_brecha(f"Exactitud balanceada — entrenamiento dentro de la muestra vs. prueba {ANIO_PRUEBA}", ba_train, ba_test)
brecha_clas_oob = ver_brecha(f"Exactitud balanceada — out-of-bag vs. prueba {ANIO_PRUEBA}", ba_oob, ba_test)
brecha_clas_cv = ver_brecha(f"Exactitud balanceada — validación por departamentos ({ANIO_INICIO}-{ANIO_PRUEBA - 1}) vs. prueba {ANIO_PRUEBA}", ba_cv_dep, ba_test)
auc_test = roc_auc_score(y_test_clas, prob_test)
print(f"AUC ({ANIO_PRUEBA}): {auc_test:.3f}  ·  AUC out-of-bag (entrenamiento): {roc_auc_score(y_train_clas, prob_oob):.3f}")
m_clas_test = metricas_clasificacion(y_test_clas, pred_test, prob_test, f"prueba {ANIO_PRUEBA}")
matriz = matriz_confusion(y_test_clas, pred_test, f"Matriz de confusión — {ANIO_PRUEBA}")

### 5.1 Curva ROC, precisión-recall, ganancia y calibración (para priorizar, no solo para clasificar)

Las mismas cuatro lecturas de la misma probabilidad predicha que en el gradient boosting, pensadas para cuatro
públicos distintos: la **ROC** para el lector de minería de datos, la **precisión-recall** para un Observatorio
con recursos limitados de verificación, la **ganancia** para el interesado que pregunta "¿cuánto cubro si reviso
solo el 20 %?" (OBJ-N1) y la **calibración** para quien quiere usar la probabilidad como número.

Una advertencia de lectura que solo aplica a los modelos con `class_weight`: si la validación eligió
`balanced_subsample`, las probabilidades se desplazan hacia la clase minoritaria y la curva de calibración se
despega de la diagonal, aunque el orden (ROC, ganancia) sea igual de bueno.

In [ ]:
cobertura_2025 = lecturas_prioridad(y_test_clas, prob_test, test.def_ideam_ha.fillna(0), f"({ANIO_PRUEBA})",
                                    f"prueba {ANIO_PRUEBA}", "Random forest", color="#c8541a")

**Lectura: qué significa este desempeño para OBJ-N1 y CDM-7.**

- **Desempeño en 2025.** AUC 0,930 y exactitud balanceada 0,854. Al umbral 0,5, de cada 10 municipios que marca, 9 sí deforestan (precisión 0,92) y deja pasar menos de 2 de cada 10 que deforestan (recall 0,83); la especificidad
  es 0,87.
- **CDM-7 se cumple en ¿deforesta?** en sus tres versiones: la brecha es 5,5 % contra el entrenamiento dentro de la muestra (la más optimista para un bosque), -1,4 % contra el *out-of-bag* y -6,3 % contra la validación por
  departamentos. Que sea negativa significa que en 2025 el modelo rinde igual o mejor que en la validación.
- **Cómo leer la curva de ganancia.** Revisando el 10 %, 20 % y 30 % de los municipios-año con mayor riesgo se cubre el 15 %, 31 % y 46 % de los que deforestan, pero el 60 %, 87 % y 95 % de las **hectáreas**. La cobertura de
  municipios casi llega al tope matemático (63/410 = 15,4 %, 127/410 = 31,0 %, 191/410 = 46,6 %), porque el 64 % de los municipios-año deforesta: los que el modelo pone arriba casi siempre sí deforestan, y lo que no se puede es
  "descartar" a los demás. El argumento para el Observatorio es el de las hectáreas: con el 30 % de la lista se cubre casi toda la deforestación.
- **Una advertencia sobre las probabilidades.** Con `balanced_subsample` y árboles sin límite de profundidad, la probabilidad se satura en 1,0 para los municipios más riesgosos (los 15 primeros de la lista de 7.4 tienen
  riesgo prácticamente igual a 1,0). Sirve para ordenar y para el umbral, pero entre esos municipios no distingue cuál es más riesgoso, y no debe leerse como "100 % de probabilidad de deforestar".

In [ ]:
en_deforesta_test = (y_test_clas == 1).values
regresor = RandomForestRegressor(**FIJOS_RF, **params_reg, oob_score=True, n_jobs=-1)
regresor.fit(X_train[en_deforesta], y_train_tasa[en_deforesta])

pred_train_tasa = regresor.predict(X_train[en_deforesta])
pred_test_tasa = regresor.predict(X_test[en_deforesta_test])
y_test_tasa_real = y_test_tasa[en_deforesta_test]

r2_train = r2_score(y_train_tasa[en_deforesta], pred_train_tasa)
r2_test_ = r2_score(y_test_tasa_real, pred_test_tasa)
r2_oob = r2_score(y_train_tasa[en_deforesta], regresor.oob_prediction_)
brecha_reg = ver_brecha(f"R² — entrenamiento dentro de la muestra vs. prueba {ANIO_PRUEBA} (solo deforesta > 0)", r2_train, r2_test_)
brecha_reg_oob = ver_brecha(f"R² — out-of-bag vs. prueba {ANIO_PRUEBA}", r2_oob, r2_test_) if r2_oob > 0 else np.nan
if not r2_oob > 0:
    print(f"R² out-of-bag = {r2_oob:.3f} (no positivo: la brecha relativa contra él no es interpretable)")
m_reg_test = metricas_regresion(y_test_tasa_real, pred_test_tasa, f"prueba {ANIO_PRUEBA}")
r2_test, mae_test, medae_test, rmse_test = m_reg_test["r2"], m_reg_test["mae"], m_reg_test["medae"], m_reg_test["rmse"]
print("(MedAE y Spearman son más robustos que MAE/RMSE/R² frente a los pocos casos extremos de 5.3)")

fig, ax = plt.subplots(1, 2, figsize=(12, 5.5))
ax[0].scatter(y_test_tasa_real, pred_test_tasa, alpha=.5, color="#c8541a")
lim = max(y_test_tasa_real.max(), pred_test_tasa.max())
ax[0].plot([0, lim], [0, lim], color="grey", ls="--", lw=1)
ax[0].set(xlabel="Tasa real (%)", ylabel="Tasa predicha (%)", title=f"Dado que deforesta, ¿cuánto? — {ANIO_PRUEBA} (escala natural)")
ax[1].scatter(y_test_tasa_real, np.clip(pred_test_tasa, 1e-4, None), alpha=.5, color="#c8541a")
ax[1].plot([1e-3, lim], [1e-3, lim], color="grey", ls="--", lw=1)
ax[1].set(xscale="log", yscale="log", xlabel="Tasa real (%, escala log)", ylabel="Tasa predicha (%, escala log)",
          title="Lo mismo en escala logarítmica")
plt.tight_layout()
plt.show()

### 5.2 Evidencia: por qué separar en clasificación + regresión condicional

El mismo experimento del gradient boosting: se entrena, solo para esta comparación, una regresión de **una sola
parte** sobre todas las filas de 2021-2024 (mezclando los ceros estructurales con la magnitud real), con los
mismos hiperparámetros ya encontrados para la condicional, y se compara **donde de verdad importa la magnitud**:
las filas de 2025 que sí deforestaron. Si la afirmación de la Tabla 26 es cierta, la de una sola parte debería
subestimar la tasa de los municipios que sí deforestan, porque los ceros "jalan" la predicción hacia abajo.

In [ ]:
regresor_directo = RandomForestRegressor(**FIJOS_RF, **params_reg, n_jobs=-1)
regresor_directo.fit(X_train, y_train_tasa)
pred_directo_def = regresor_directo.predict(X_test)[en_deforesta_test]  # solo donde SI hubo deforestacion real

comparacion = pd.DataFrame([
    {"enfoque": "Una sola regresión (entrenada con TODAS las filas, ceros incluidos)",
     "R²": r2_score(y_test_tasa_real, pred_directo_def), "MAE": mean_absolute_error(y_test_tasa_real, pred_directo_def),
     "predicción media": pred_directo_def.mean()},
    {"enfoque": "Regresión condicional (entrenada solo con deforesta == 1, este notebook)",
     "R²": r2_test, "MAE": mae_test, "predicción media": pred_test_tasa.mean()},
])
print(f"Evaluado solo sobre las {en_deforesta_test.sum()} filas de {ANIO_PRUEBA} que SÍ deforestaron "
      f"(tasa real promedio: {y_test_tasa_real.mean():.3f}):")
comparacion.round(3)

In [ ]:
mejora = comparacion.loc[1, "predicción media"] - comparacion.loc[0, "predicción media"]
real = y_test_tasa_real.mean()
print(f"La predicción media sobre los municipios que sí deforestaron es {comparacion.loc[0, 'predicción media']:.3f} con una sola "
      f"regresión y {comparacion.loc[1, 'predicción media']:.3f} con la condicional (la real es {real:.3f}).")
if mejora > 0:
    print("Los ceros estructurales sí jalan la predicción hacia abajo en la regresión de una sola parte: la condicional "
          "queda más cerca de la tasa real, que es la razón de separar los dos modelos.")
else:
    print("En este bosque la separación no acerca la predicción media a la real: conviene decirlo tal cual en el documento.")
print("(Aparte: cuando el clasificador comete un falso positivo, el modelo de dos partes le asigna una tasa > 0 a un "
      "municipio que no deforestó. Es un error distinto, que ya queda reflejado en la precisión y la especificidad de la sección 5.)")

### 5.3 Por qué el R² de "¿cuánto?" es tan sensible a un solo municipio (y qué lecturas no lo son)

El R² es un promedio de errores al cuadrado: una sola fila con un error enorme pesa como decenas con error
normal. **El caso es Pedraza (Magdalena)**: su bosque IDEAM es de apenas ~9 ha, y como la tasa se calcula sobre el
bosque que queda (`def_ideam_ha / bosque_ideam_ha`), perder 8 ha se vuelve una tasa de 88 %, muy por fuera de la
escala de cualquier otro municipio rural. Ninguna de las 15 variables distingue "este municipio casi no tiene
bosque" de los demás. Como en el gradient boosting, el R² reportado **no recorta ni borra la fila**; el cálculo
sin Pedraza es solo un diagnóstico de sensibilidad.

Además se miran dos lecturas que no dependen de ese extremo: el R² en **escala logarítmica** (¿el modelo acierta
en órdenes de magnitud?) y la **correlación de Spearman** (¿ordena bien a los municipios, del que menos al que
más deforesta?). Para un Observatorio que prioriza, ordenar bien suele importar más que acertar la tasa exacta.

In [ ]:
test_def = test[en_deforesta_test]
es_pedraza = (test_def.cod_dane == "47541").values

if es_pedraza.any():
    print(f"Tasa real de Pedraza ({ANIO_PRUEBA}): {y_test_tasa_real.values[es_pedraza][0]:.1f} %  ·  "
          f"predicha por el modelo: {pred_test_tasa[es_pedraza][0]:.1f} %")

    sin_pedraza = ~es_pedraza
    r2_sin_pedraza = r2_score(y_test_tasa_real.values[sin_pedraza], pred_test_tasa[sin_pedraza])
    mae_sin_pedraza = mean_absolute_error(y_test_tasa_real.values[sin_pedraza], pred_test_tasa[sin_pedraza])
    print()
    brecha_sin_pedraza = ver_brecha("R² excluyendo solo a Pedraza (diagnóstico)", r2_train, r2_sin_pedraza)
    print(f"Error absoluto promedio excluyendo a Pedraza: {mae_sin_pedraza:.3f} (vs. {mae_test:.3f} con Pedraza incluido)")
    print(f"\nLecturas robustas, con Pedraza incluido: R² en escala log = {m_reg_test['r2_log']:.3f}  ·  "
          f"Spearman = {m_reg_test['spearman']:.3f}")
else:
    print("Pedraza no está entre los municipios que deforestaron en el año de prueba: se omite este diagnóstico.")
    r2_sin_pedraza = mae_sin_pedraza = brecha_sin_pedraza = np.nan

**Lectura: qué sí y qué no se puede decir de "¿cuánto?" (CDM-7).**

El R² de prueba es 0,062 con una brecha de 83 % contra el entrenamiento (77,7 % contra el *out-of-bag*): **no cumple CDM-7**, y la causa es un solo municipio. Sin Pedraza, el R² de prueba sube a 0,368 y la brecha cae a 1,4 %, que sí cumple.
Aun así, el bosque **ordena** la magnitud (Spearman 0,73; R² en escala logarítmica 0,41), es decir, acierta en órdenes de magnitud y en qué
municipio deforesta más, aunque no acierte el valor exacto. Por eso para el Observatorio el "¿cuánto?" sirve para **ordenar por severidad esperada**, no para estimar hectáreas. Con departamentos fuera, esa capacidad baja (Spearman 0,61, R² en
escala log 0,17): los saltos grandes y poco frecuentes del corredor seco del Caribe siguen sin tener ninguna variable que los distinga. Es una limitación para declarar tal cual en el documento.

### 5.4 ¿Qué aportan el crédito y la producción más allá de lo estructural?

El gradient boosting reporta el modelo completo; no dice cuánto del desempeño se debe a lo que ya sabe cualquiera
(el municipio tiene bosque y es grande) y cuánto a las variables financieras y productivas que son el motivo del
proyecto. Esta comparación lo separa, con los mismos hiperparámetros y evaluando en 2025:

| Especificación | Variables |
|---|---|
| **Persistencia** (línea base, sin modelo) | La frecuencia con que deforestaron en el entrenamiento los municipios que sí / no deforestaron el año anterior |
| **A. Estructural** | `bosque`, `area` |
| **B. Principal** | Las 15 explicativas (el modelo de este notebook) |
| **C. Con persistencia** | B + si el municipio deforestó el año anterior |
| **D. Con región** | B + la región (categórica), que la versión anterior incluía |

C necesita la deforestación del año anterior, que no existe para 2021 (el IDEAM no tiene tasa 2020 en el panel), así que
**todas las especificaciones de esta sección se entrenan con 2022-2024** para ser comparables entre sí. Por eso B
aquí no coincide exactamente con el modelo de la sección 5 (que entrena con 2021-2024).

In [ ]:
# Rezago de un año de "deforesta" (no existe para 2021) y region, solo para esta seccion
panel["deforesta_l1"] = panel.groupby("cod_dane").tasa.transform(
    lambda s: (s.shift(1) > 0).astype(float).where(s.shift(1).notna()))
extra = (panel[["cod_dane", "anio", "deforesta_l1"]]
         .merge(anual[["cod_dane", "anio", "region"]], on=["cod_dane", "anio"], how="left"))
train_e = train.merge(extra, on=["cod_dane", "anio"], how="left")
test_e = test.merge(extra, on=["cod_dane", "anio"], how="left")
train_c = train_e[train_e.anio >= ANIO_INICIO + 1].reset_index(drop=True)


def ajustar_rf_spec(cols_num, cols_cat=()):
    pasos = [("num", "passthrough", list(cols_num))]
    if cols_cat:
        pasos.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), list(cols_cat)))
    modelo = Pipeline([("pre", ColumnTransformer(pasos)),
                       ("rf", RandomForestClassifier(**FIJOS_RF, **params_clas, n_jobs=-1))])
    return modelo.fit(train_c[list(cols_num) + list(cols_cat)], train_c.deforesta)


especificaciones = {"A. Estructural": (["bosque", "area"], ()),
                    "B. Principal (15 variables)": (EXPLICATIVAS, ()),
                    "C. Con persistencia": (EXPLICATIVAS + ["deforesta_l1"], ()),
                    "D. Con región": (EXPLICATIVAS, ("region",))}
tasa_cond = train_c.groupby("deforesta_l1").deforesta.mean()
prob_pers = test_e.deforesta_l1.map(tasa_cond).values

filas = [{"especificación": "Persistencia (año anterior, sin modelo)", "AUC": roc_auc_score(y_test_clas, prob_pers),
          "exactitud balanceada": balanced_accuracy_score(y_test_clas, prob_pers >= UMBRAL),
          "AP": average_precision_score(y_test_clas, prob_pers)}]
for nombre, (num, cat) in especificaciones.items():
    mod = ajustar_rf_spec(num, cat)
    prob = mod.predict_proba(test_e[list(num) + list(cat)])[:, 1]
    filas.append({"especificación": nombre, "AUC": roc_auc_score(y_test_clas, prob),
                  "exactitud balanceada": balanced_accuracy_score(y_test_clas, prob >= UMBRAL),
                  "AP": average_precision_score(y_test_clas, prob)})
especificaciones_tabla = pd.DataFrame(filas).set_index("especificación")
especificaciones_tabla.round(4)

In [ ]:
e = especificaciones_tabla.AUC
print(f"El bosque y el área solos (A) logran un AUC de {e['A. Estructural']:.3f}; con las 15 variables (B) sube a "
      f"{e['B. Principal (15 variables)']:.3f}: el crédito, los cultivos y el ganado aportan {e['B. Principal (15 variables)'] - e['A. Estructural']:+.3f} de AUC "
      f"sobre lo estructural.")
print(f"Saber si el municipio deforestó el año pasado (C) lleva el AUC a {e['C. Con persistencia']:.3f} "
      f"({e['C. Con persistencia'] - e['B. Principal (15 variables)']:+.3f} sobre B); la persistencia sola, sin ningún modelo, da {e['Persistencia (año anterior, sin modelo)']:.3f}.")
print(f"Agregar la región (D) cambia el AUC en {e['D. Con región'] - e['B. Principal (15 variables)']:+.3f}: "
      f"{'la región no agrega información que las 15 variables no tengan' if abs(e['D. Con región'] - e['B. Principal (15 variables)']) < 0.005 else 'la región sí agrega algo de información'}.")

**Lectura: qué aportan el crédito y la producción, y qué no (ODM-3, OBJ-N1).**

- Solo con bosque y área (A) el AUC es 0,917; con las 15 variables (B), 0,929: **crédito, cultivos y ganado agregan +0,011 de AUC** sobre lo estructural. Es un aporte pequeño pero medible.
- **La persistencia no ayuda aquí** (C: -0,005): saber si el municipio deforestó el año pasado no agrega nada que las 15 variables no tengan. La persistencia sola, sin ningún modelo, da 0,827; la región tampoco aporta.
- **Qué significa para el objetivo.** La capacidad de priorizar viene sobre todo de dónde hay bosque y de qué tamaño es el municipio (la variable objetivo "deforesta > 0" favorece mecánicamente a los que tienen mucho bosque). Las variables
  financieras y productivas son un complemento, no el motor de la predicción. El aporte del modelo al Observatorio es (1) una priorización territorial confiable y (2) una lectura de **qué asociaciones financiero-productivas existen** (ODM-3),
  no la afirmación de que el crédito predice la deforestación. Que el aporte predictivo del crédito sea pequeño no prueba que sea irrelevante: las variables son del mismo año, están correlacionadas y se miden por km².

## 6. Evaluación dejando departamentos completos fuera

La sección 5 mide generalización en el tiempo (entrena 2021-2024, prueba 2025), pero los 639 municipios de 2025
son los mismos que el modelo vio entrenando. Falta la otra mitad del supuesto de independencia: medir
generalización en el **territorio**, dejando fuera departamentos completos, porque municipios vecinos se parecen
(mismo bosque, mismas vías, mismo frente de colonización).

Igual que en el gradient boosting: `GroupKFold` por departamento (5 pliegues) sobre **todo 2021-2025**, con los
hiperparámetros ya elegidos en la sección 4 (sin volver a buscar). Cada fila queda predicha por un modelo que
nunca vio su departamento: es la prueba de que el modelo sirve para priorizar territorios nuevos, que es lo que
necesita el Observatorio (OBJ-N1).

In [ ]:
m = muestra.reset_index(drop=True)
grupos_todos = m.nombre_departamento
gkf_eval = GroupKFold(n_splits=5)

pred_clas_oof = pd.Series(np.nan, index=m.index)
prob_clas_oof = pd.Series(np.nan, index=m.index)
pred_tasa_oof = pd.Series(np.nan, index=m.index)

for i_dentro, i_fuera in gkf_eval.split(m[EXPLICATIVAS], m.deforesta, groups=grupos_todos):
    clas_fold = RandomForestClassifier(**FIJOS_RF, **params_clas, n_jobs=-1)
    clas_fold.fit(m[EXPLICATIVAS].iloc[i_dentro], m.deforesta.iloc[i_dentro])
    pred_clas_oof.iloc[i_fuera] = clas_fold.predict(m[EXPLICATIVAS].iloc[i_fuera])
    prob_clas_oof.iloc[i_fuera] = clas_fold.predict_proba(m[EXPLICATIVAS].iloc[i_fuera])[:, 1]

    en_def_dentro = (m.deforesta.iloc[i_dentro] == 1).values
    reg_fold = RandomForestRegressor(**FIJOS_RF, **params_reg, n_jobs=-1)
    reg_fold.fit(m[EXPLICATIVAS].iloc[i_dentro][en_def_dentro], m.tasa.iloc[i_dentro][en_def_dentro])
    pred_tasa_oof.iloc[i_fuera] = reg_fold.predict(m[EXPLICATIVAS].iloc[i_fuera])

en_def_oof = (m.deforesta == 1).values
ba_departamentos = balanced_accuracy_score(m.deforesta, pred_clas_oof)

print(f"Departamentos en la muestra: {grupos_todos.nunique()}")
print(f"Exactitud balanceada, ¿deforesta? (departamentos fuera): {ba_departamentos:.3f}  "
      f"(referencia, prueba temporal {ANIO_PRUEBA}: {ba_test:.3f})")
m_clas_oof = metricas_clasificacion(m.deforesta, pred_clas_oof, prob_clas_oof, "departamentos fuera")

m_reg_oof = metricas_regresion(m.tasa[en_def_oof], pred_tasa_oof[en_def_oof], "departamentos fuera")
r2_departamentos, mae_departamentos = m_reg_oof["r2"], m_reg_oof["mae"]
medae_departamentos, rmse_departamentos = m_reg_oof["medae"], m_reg_oof["rmse"]
print(f"(referencia, prueba temporal {ANIO_PRUEBA} — R²: {r2_test:.3f}  ·  MAE: {mae_test:.3f}  ·  "
      f"MedAE: {medae_test:.3f}  ·  RMSE: {rmse_test:.3f})")

### 6.1 Matriz de confusión, ROC, precisión-recall, ganancia y calibración (departamentos fuera)

Las mismas lecturas de 5/5.1 pero con las predicciones "limpias" de la sección 6: es la versión más exigente de
cada gráfica, la que de verdad responde si el modelo sirve para un departamento que el Observatorio todavía no ha
mirado (OBJ-N1).

In [ ]:
matriz_oof = matriz_confusion(m.deforesta, pred_clas_oof, "Matriz de confusión — departamentos fuera")
cobertura_oof = lecturas_prioridad(m.deforesta, prob_clas_oof, m.def_ideam_ha.fillna(0), "(departamentos fuera)",
                                   "sin que el modelo haya visto su departamento", "Random forest", color="#c8541a")

**Lectura: lo que cambia cuando el territorio es nuevo (OBJ-N1).**

La prueba temporal 2025 es la más favorable para un bosque, porque los 639 municipios ya estuvieron en el entrenamiento (en años anteriores). Cuando se dejan departamentos completos fuera, el desempeño baja: AUC de 0,930 a 0,905, exactitud
balanceada de 0,854 a 0,817 y especificidad de 0,873 a 0,796 (el recall se mantiene: 0,834 frente a 0,838). Es decir, en un departamento nuevo el modelo **no deja pasar más deforestación, pero sí da más falsas alarmas**. En hectáreas, revisando el
10 % más riesgoso se cubre el 55 % (frente al 60 % de la prueba temporal); con el 30 %, el 95 % en ambos casos. Esta es la cifra que corresponde al caso real del Observatorio (priorizar donde todavía no ha mirado); la prueba de 2025 sola sobrestima ese caso.

### 6.2 Los errores más grandes de "¿cuánto?" (departamentos fuera)

Igual que en 5.3, vale la pena mirar qué filas concretas producen el R². Pedraza vuelve a aparecer (es la misma
fila, ahora predicha por un modelo que tampoco vio su departamento); lo que importa ver es si los siguientes
errores son también casos de bosque casi nulo o saltos reales poco frecuentes que ningún modelo con estas
variables podría anticipar.

In [ ]:
errores = pd.DataFrame({
    "cod_dane": m.cod_dane[en_def_oof], "municipio": m.nombre_municipio[en_def_oof], "departamento": m.nombre_departamento[en_def_oof],
    "anio": m.anio[en_def_oof], "tasa_real": m.tasa[en_def_oof], "tasa_predicha": pred_tasa_oof[en_def_oof],
})
errores["error_absoluto"] = (errores.tasa_real - errores.tasa_predicha).abs()
errores = errores.sort_values("error_absoluto", ascending=False)
print("Las 6 filas con mayor error absoluto (departamentos fuera):")
errores.head(6).round(2)

### 6.3 Riesgo predicho por departamento (lectura de negocio)

Las probabilidades de la sección 6 ya son "limpias": cada departamento quedó predicho por un modelo que nunca lo
vio entrenar, así que se pueden agregar y leer como una priorización territorial para el Observatorio (OBJ-N1).
Se muestran los diez departamentos con mayor probabilidad promedio según el modelo, junto con la frecuencia real
de municipios-año que deforestaron, para ver si la priorización coincide con lo que efectivamente pasó.

In [ ]:
por_depto = pd.DataFrame({
    "riesgo_predicho": prob_clas_oof.groupby(m.nombre_departamento).mean(),
    "tasa_deforesta_real": m.groupby("nombre_departamento").deforesta.mean(),
    "municipios_anio": m.groupby("nombre_departamento").size(),
}).sort_values("riesgo_predicho", ascending=False).head(10)

fig, ax = plt.subplots(figsize=(9, 5.5))
y_pos = np.arange(len(por_depto))
ax.barh(y_pos - 0.2, por_depto.riesgo_predicho, height=0.4, color="#c8541a", label="Riesgo predicho (modelo)")
ax.barh(y_pos + 0.2, por_depto.tasa_deforesta_real, height=0.4, color="#2a78d6", label="% de municipios-año que sí deforestaron")
ax.set_yticks(y_pos, por_depto.index)
ax.invert_yaxis()
ax.set(xlabel="Proporción", title="Los 10 departamentos de mayor riesgo predicho (OBJ-N1, con departamentos fuera)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, fontsize=10, frameon=False)
plt.tight_layout()
plt.show()
por_depto.round(3)

## 7. Importancia de variables (ODM-3)

Qué líneas de crédito y qué cadenas pesan más, controlando por las demás: la tercera pregunta específica de
negocio y la que responde directamente ODM-3.

**Una aclaración necesaria, igual que en el gradient boosting: no todas las 15 variables son del mismo tipo.**
`bosque` y `area` son **estructurales** (describen cómo es el municipio; nadie las puede "mover" con una política).
Las otras 13 —crédito, cultivos, ganado— son **variables de intervención**: reflejan decisiones de financiamiento
y producción que FINAGRO, el Observatorio o un gremio sí pueden influir. Un ranking que las mezcle puede leerse
mal como "qué palanca mover", por eso la comparación se separa en dos preguntas (7.1).

**Tres medidas, para no depender de una sola.** (1) La **importancia por impureza** (MDI), la contraparte de la
"ganancia" del gradient boosting: cuánto reduce cada variable la impureza en el entrenamiento; favorece a las
variables continuas con muchos valores. (2) La **permutación** sobre 2025, que no se deja engañar por eso:
cuánto empeora el modelo al barajar la variable. (3) La **permutación por bloques y actividades** (7.5), que
baraja juntas las variables correlacionadas. Cuando las tres cuentan la misma historia, es confiable.

In [ ]:
ESTRUCTURALES = {"bosque", "area"}
POLITICA = [v for v in EXPLICATIVAS if v not in ESTRUCTURALES]
CREDITO = ["cred", "cred_gan", "cred_palma", "cred_cacao", "cred_cafe", "cred_arroz"]
nombres_politica = [NOMBRES[v] for v in POLITICA]
nombres_credito = [NOMBRES[v] for v in CREDITO]

# Bloques y actividades: los mismos en los tres modelos. Se barajan JUNTAS todas las columnas del grupo,
# para no subestimar variables que se reparten la importancia porque estan correlacionadas entre si
# (credito y area de cafe, credito ganadero y bovinos/fincas...).
BLOQUES = {"Crédito (6 líneas)": CREDITO,
           "Cultivos (5 columnas de % del área)": ["agricola", "palma", "cacao", "cafe", "arroz"],
           "Ganado (bovinos y fincas)": ["bovinos", "fincas"],
           "Estructura (bosque y área)": ["bosque", "area"]}
ACTIVIDADES = {"Ganadería (crédito + bovinos + fincas)": ["cred_gan", "bovinos", "fincas"],
               "Café (crédito + área)": ["cred_cafe", "cafe"],
               "Cacao (crédito + área)": ["cred_cacao", "cacao"],
               "Palma (crédito + área)": ["cred_palma", "palma"],
               "Arroz (crédito + área)": ["cred_arroz", "arroz"],
               "Escala total (crédito y área cosechada totales)": ["cred", "agricola"]}


def auc_rapido(y, p):
    # AUC por rangos (Mann-Whitney); equivale a roc_auc_score pero es ~10 veces mas rapido, y se usa miles de veces
    r = stats.rankdata(p)
    n1 = y.sum()
    n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def permutacion_con_p(modelo, X, y, tipo, grupos, usadas=None, n_repeticiones=30, n_perm_boot=10, n_boot=400):
    # Importancia por permutacion sobre 2025, con DOS valores p:
    #  - p_valor: el del gradient boosting (prueba t de las 30 repeticiones contra cero). Solo recoge el azar del
    #    barajado, no la incertidumbre de tener 639 municipios: es optimista, y se deja para poder comparar.
    #  - p_territorio: bootstrap por DEPARTAMENTO (se remuestrean departamentos completos, como en los errores
    #    robustos por conglomerado espacial que pide CDM-5). Es la prueba que se usa para contar variables.
    # p_holm corrige p_territorio por comparaciones multiples (Holm) entre las variables que el modelo usa.
    scoring, predecir, metrica = (("roc_auc", lambda mod, Z: mod.predict_proba(Z)[:, 1], auc_rapido) if tipo == "clas"
                                  else ("r2", lambda mod, Z: mod.predict(Z), r2_score))
    y = np.asarray(y)
    resultado = permutation_importance(modelo, X, y, n_repeats=n_repeticiones, random_state=SEMILLA,
                                        scoring=scoring, n_jobs=1)
    rng = np.random.default_rng(SEMILLA)
    p0 = predecir(modelo, X)
    perms = {}
    for col in X.columns:
        ps = []
        for _ in range(n_perm_boot):
            Xp = X.copy()
            Xp[col] = X[col].values[rng.permutation(len(X))]
            ps.append(predecir(modelo, Xp))
        perms[col] = np.array(ps)
    g = np.asarray(grupos)
    deptos = np.unique(g)
    idx_depto = {d: np.where(g == d)[0] for d in deptos}
    caidas = {col: [] for col in X.columns}
    for _ in range(n_boot):
        idx = np.concatenate([idx_depto[d] for d in rng.choice(deptos, size=len(deptos), replace=True)])
        base = metrica(y[idx], p0[idx])
        for col in X.columns:
            caidas[col].append(base - np.mean([metrica(y[idx], pp[idx]) for pp in perms[col]]))
    filas = []
    for i, var in enumerate(X.columns):
        est = metrica(y, p0) - np.mean([metrica(y, pp) for pp in perms[var]])
        se = np.std(caidas[var], ddof=1)
        t_p = stats.ttest_1samp(resultado.importances[i], 0).pvalue
        filas.append({"variable": NOMBRES[var], "clave": var, "importancia_media": resultado.importances_mean[i],
                      "desv": resultado.importances_std[i], "p_valor": 1.0 if np.isnan(t_p) else t_p,
                      "ic95_inf_territorio": est - 1.96 * se, "ic95_sup_territorio": est + 1.96 * se,
                      "p_territorio": 1.0 if (se == 0 or np.isnan(se)) else stats.norm.sf(est / se)})
    t = pd.DataFrame(filas).sort_values("importancia_media", ascending=False).reset_index(drop=True)
    usadas = list(X.columns) if usadas is None else list(usadas)
    t["p_holm"] = np.nan
    en_uso = t.clave.isin(usadas)
    t.loc[en_uso, "p_holm"] = multipletests(t.loc[en_uso, "p_territorio"], method="holm")[1]
    t["tipo"] = np.where(t.clave.isin(ESTRUCTURALES), "estructural", "intervención")
    return t


def permutar_bloques(modelo, X, y, grupos, puntaje, n_repeats=30, semilla=SEMILLA):
    # Caida de la metrica al desordenar juntas (con la misma permutacion de filas) las columnas de cada grupo.
    rng = np.random.default_rng(semilla)
    base = puntaje(modelo, X, y)
    filas = []
    for nombre, cols in grupos.items():
        caidas = []
        for _ in range(n_repeats):
            Xp = X.copy()
            Xp[cols] = X[cols].values[rng.permutation(len(X))]
            caidas.append(base - puntaje(modelo, Xp, y))
        filas.append({"grupo": nombre, "n_variables": len(cols), "caida": np.mean(caidas), "desv": np.std(caidas),
                      "p_valor": stats.ttest_1samp(caidas, 0).pvalue})
    return pd.DataFrame(filas).set_index("grupo").sort_values("caida", ascending=False)


def puntaje_auc(modelo, X, y):
    return roc_auc_score(y, modelo.predict_proba(X)[:, 1])


def puntaje_r2(modelo, X, y):
    return r2_score(y, modelo.predict(X))


def graficar_bloques(tablas, titulo_metrica):
    fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
    for eje, (clave, titulo) in zip(ax, [("bloques", "Por bloque temático"), ("actividades", "Por actividad productiva")]):
        t = tablas[clave].sort_values("caida")
        eje.barh(t.index, t.caida, xerr=t.desv, color="#2a78d6", capsize=3)
        eje.axvline(0, ls="--", color="grey")
        eje.set_xlabel(f"caída de {titulo_metrica} al barajar el grupo ({ANIO_PRUEBA})")
        eje.set_title(titulo)
    plt.tight_layout()
    plt.show()


from matplotlib.patches import Patch
COLORES_POR_NOMBRE = {NOMBRES[v]: ("#9aa0a6" if v in ESTRUCTURALES else "#c8541a") for v in EXPLICATIVAS}


def graficar_importancia(modelo, titulo):
    imp = pd.Series(modelo.feature_importances_, index=EXPLICATIVAS).rename(NOMBRES).sort_values()
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(imp.index, imp.values, color=[COLORES_POR_NOMBRE[n] for n in imp.index])
    ax.set(title=titulo, xlabel="Importancia (reducción de impureza, MDI)")
    ax.legend(handles=[Patch(color="#9aa0a6", label="Estructural (no es palanca de política)"),
                        Patch(color="#c8541a", label="Crédito / producción (sí es palanca)")], loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.show()
    return imp


imp_clas = graficar_importancia(clasificador, "Importancia — ¿deforesta?")
imp_reg = graficar_importancia(regresor, "Importancia — dado que deforesta, ¿cuánto?")
print("Top 5 — ¿deforesta? (MDI, escala 0-1):")
for var, val in imp_clas.sort_values(ascending=False).head(5).items():
    print(f"  {var:42s} {val:.3f}")
print("\nTop 5 — dado que deforesta, ¿cuánto?:")
for var, val in imp_reg.sort_values(ascending=False).head(5).items():
    print(f"  {var:42s} {val:.3f}")

### 7.1 Dos preguntas distintas, dos lecturas distintas

**Pregunta 1 — ¿qué tan arriesgado es el municipio, en general?** Aquí `bosque` y `area` mandan, y no es un
problema: son la variable de **exposición** (un municipio con más bosque tiene más por perder; uno más grande y
remoto es más difícil de vigilar). Confirma que el modelo prioriza con el criterio correcto (OBJ-N1).

**Pregunta 2 — entre las variables que el Observatorio sí puede incidir, ¿cuál pesa más?** Esta es la que
responde ODM-3 y la que le importa a un cliente que quiere saber "¿qué cadena debería mirar primero?". Para
responderla se comparan las 13 variables de intervención **entre sí**, y luego solo las seis líneas de crédito.

In [ ]:
imp_clas_politica = imp_clas.loc[nombres_politica].sort_values(ascending=False)
imp_reg_politica = imp_reg.loc[nombres_politica].sort_values(ascending=False)
imp_clas_credito = imp_clas.loc[nombres_credito].sort_values(ascending=False)
imp_reg_credito = imp_reg.loc[nombres_credito].sort_values(ascending=False)

for titulo, serie in [("Ranking entre las 13 variables de intervención — ¿deforesta?", imp_clas_politica),
                      ("Ranking entre las 13 variables de intervención — ¿cuánto?", imp_reg_politica),
                      ("SOLO entre las 6 líneas de crédito — ¿deforesta?", imp_clas_credito),
                      ("SOLO entre las 6 líneas de crédito — ¿cuánto?", imp_reg_credito)]:
    print(titulo + ":")
    for var, val in serie.items():
        print(f"  {var:42s} {val:.3f}")
    print()

print(f"Respuesta directa a ODM-3 (\"¿qué línea de crédito pesa más?\"): en ¿deforesta?, {imp_clas_credito.index[0]}; "
      f"en ¿cuánto?, {imp_reg_credito.index[0]}. Las que menos aportan: {imp_clas_credito.index[-1]} y {imp_reg_credito.index[-1]}.")
print(f"La variable de intervención que más pesa (no crédito) en ¿deforesta? es {imp_clas_politica.index[0]}.")

### 7.2 Variables con asociación significativa (CDM-5), con corrección por comparaciones múltiples

CDM-5 pide "variables financieras y de producción con asociación estadísticamente significativa (p < 0,05,
**corregida por comparaciones múltiples**)", con "errores estándar robustos por conglomerado espacial". La
importancia por impureza no trae valor p. Como en el gradient boosting, aquí se responde **desde el modelo** con la
importancia por permutación sobre 2025 (nunca usada para entrenar), pero con tres precisiones:

1. **`p_valor`** es el del gradient boosting: una prueba t sobre las 30 repeticiones del barajado. Solo recoge el azar
   del barajado, no la incertidumbre de haber observado 639 municipios, así que es optimista (casi todo sale
   "significativo"). Se deja para poder comparar con aquél.
2. **`p_territorio`** es la prueba que se usa para contar: un *bootstrap por departamento* (se remuestrean
   departamentos completos con reemplazo, 400 veces), que es el equivalente, para un modelo de árboles, de los
   errores robustos por conglomerado espacial que pide CDM-5.
3. **`p_holm`** corrige `p_territorio` por comparaciones múltiples (Holm) entre las 15 variables.

Y una distinción que el gradient boosting no hace y que aquí importa: el "top 3" de importancia suele estar ocupado
por las variables **estructurales** (bosque, área), que no son financieras ni productivas. CDM-5 habla de variables
financieras y de producción, así que se cuentan **solo las de intervención**.

In [ ]:
perm_clas = permutacion_con_p(clasificador, X_test, y_test_clas, "clas", test.nombre_departamento)
perm_reg = permutacion_con_p(regresor, X_test[en_deforesta_test], y_test_tasa_real, "reg",
                             test.nombre_departamento[en_deforesta_test])

cols = ["variable", "tipo", "importancia_media", "p_valor", "p_territorio", "p_holm"]
print(f"¿deforesta? — permutación sobre {ANIO_PRUEBA} (caída del AUC), 6 primeras:")
display(perm_clas[cols].head(6).round(4))
print(f"Dado que deforesta, ¿cuánto? — permutación sobre {ANIO_PRUEBA} (caída del R²), 6 primeras:")
display(perm_reg[cols].head(6).round(4))

def contar_cdm5(perm):
    inter = perm[(perm.tipo == "intervención") & (perm.importancia_media > 0)]
    return {"top3_sig": int((perm.head(3).p_holm < 0.05).sum()),
            "interv_sig_holm": int((inter.p_holm < 0.05).sum()),
            "interv_sig_territorio": int((inter.p_territorio < 0.05).sum()),
            "interv_sig_gb": int((inter.p_valor < 0.05).sum()),
            "nombres": inter[inter.p_holm < 0.05].variable.tolist()}

cdm5_clas, cdm5_reg = contar_cdm5(perm_clas), contar_cdm5(perm_reg)
for etq, r in [("¿deforesta?", cdm5_clas), ("¿cuánto?", cdm5_reg)]:
    print(f"CDM-5 — {etq}: de las 13 variables de intervención, {r['interv_sig_gb']} tienen p < 0,05 con la prueba t del gradient "
          f"boosting, {r['interv_sig_territorio']} con el bootstrap por departamento y {r['interv_sig_holm']} tras la corrección de Holm; "
          f"de las 3 primeras del ranking general, {r['top3_sig']} (Holm).")
    if r["nombres"]:
        print("   Con p corregido < 0,05:", "; ".join(r["nombres"]))

**Lectura: qué soporta cada resultado frente a CDM-5 y ODM-3.**

- **¿Deforesta? — CDM-5 se cumple con la prueba exigente.** Tras el bootstrap por departamento y la corrección de Holm, tres variables de intervención resisten: **fincas ganaderas por 100 km²** (la de mayor aporte, caída de AUC de 0,023),
  **cacao (% del municipio)** (p Holm = 0,017) y **crédito ganadero** (p Holm = 0,025). Con la prueba t simple del barajado parecerían 11 de 13, y con el bootstrap sin corregir, 5: la diferencia entre 11 y 3 es lo que cuesta ser riguroso con
  31 departamentos y 13 variables.
- **No usar la importancia por impureza (MDI) sola.** Por MDI la línea de crédito de mayor peso es la del café, pero su permutación no resiste el bootstrap por departamento (p = 0,098; Holm = 0,77): el MDI favorece a las variables continuas con muchos valores
  distintos. La lectura confiable es la de la permutación con Holm.
- **¿Cuánto? — ninguna variable de intervención resiste (0 tras Holm).** Con R² de 0,06 en 2025 y un solo municipio dominando el error, el bosque no tiene suficiente señal para atribuir la magnitud a ninguna variable. Es una cifra que se reporta
  tal cual.
- **Asociación, no causalidad.** La importancia dice cuánto necesita el modelo cada variable para predecir, no que la variable cause la deforestación. Además, fincas ganaderas por 100 km² se mueve con el área y el bosque (tiende a ser más alta en municipios
  pequeños), así que parte de lo que mide es el tamaño del municipio.

### 7.3 Dirección del efecto (SHAP)

Todo lo anterior responde "cuánto pesa", no "en qué dirección": ¿más crédito a café se asocia con **más** o con
**menos** riesgo? Es lo que pide el objetivo de minería ("mediante modelos explicables", REQ-04) y la parte de
**magnitud, dirección e incertidumbre** de ODM-3. Se usa SHAP (`TreeExplainer`, exacto para árboles) sobre 2025:
para cada municipio-año y variable, SHAP reparte la diferencia entre su predicción y el promedio del modelo; con
signo, así que un aporte positivo empuja hacia más riesgo y uno negativo hacia menos.

- **Enjambre (summary plot).** Cada punto es un municipio-año; el color es el valor de la variable (azul bajo,
  rojo alto) y la posición horizontal, cuánto empujó esa variable la predicción de ese municipio.
- **Dependencia.** Para las tres líneas de crédito que más pesan según 7.2, el aporte SHAP contra el valor real del crédito.
- **Dependencia parcial (7.6)** complementa esto con una lectura distinta (qué hace el promedio del modelo al mover
  una variable), que no depende de SHAP.

In [ ]:
def shap_clase_1(explicador, X):
    # Segun la version de shap, un clasificador de scikit-learn devuelve una lista (una matriz por clase) o un
    # arreglo de 3 dimensiones; en los dos casos nos quedamos con la clase "deforesta".
    v = explicador.shap_values(X, check_additivity=False)
    if isinstance(v, list):
        v = v[1]
    v = np.asarray(v)
    return v[:, :, 1] if v.ndim == 3 else v


nombres_cols = [NOMBRES[c] for c in X_test.columns]
explicador_clas = shap.TreeExplainer(clasificador)
shap_clas = shap_clase_1(explicador_clas, X_test)
explicador_reg = shap.TreeExplainer(regresor)
shap_reg = np.asarray(explicador_reg.shap_values(X_test[en_deforesta_test], check_additivity=False))

shap.summary_plot(shap_clas, X_test, feature_names=nombres_cols, show=False, plot_size=(9, 6))
plt.title(f"SHAP — ¿deforesta? ({ANIO_PRUEBA})")
plt.tight_layout()
plt.show()

shap.summary_plot(shap_reg, X_test[en_deforesta_test], feature_names=nombres_cols, show=False, plot_size=(9, 6))
plt.title(f"SHAP — dado que deforesta, ¿cuánto? ({ANIO_PRUEBA})")
plt.tight_layout()
plt.show()

In [ ]:
TOP_CREDITO = perm_clas[perm_clas.clave.isin(CREDITO)].head(3).clave.tolist()  # las 3 lineas de credito que mas pesan (7.2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, var in zip(axes, TOP_CREDITO):
    shap.dependence_plot(var, shap_clas, X_test, interaction_index=None, ax=ax, show=False)
    ax.set(xlabel=NOMBRES[var], ylabel="Aporte SHAP a la predicción (¿deforesta?)", title=NOMBRES[var])
plt.suptitle(f"Dirección del efecto del crédito — ¿deforesta? ({ANIO_PRUEBA})")
plt.tight_layout()
plt.show()

print("Dirección del efecto (correlación de Spearman entre el valor de la variable y su aporte SHAP; "
      "positiva = a más variable, más riesgo; negativa = a más variable, menos riesgo):")
direccion = {}
for var in TOP_CREDITO:
    idx = list(X_test.columns).index(var)
    rho, p = stats.spearmanr(X_test[var], shap_clas[:, idx])
    direccion[var] = (rho, p)
    print(f"  {NOMBRES[var]:42s} rho = {rho:+.3f}  (p = {p:.1e})")
neg = [NOMBRES[v] for v, (r, p) in direccion.items() if r < 0 and p < 0.05]
pos = [NOMBRES[v] for v, (r, p) in direccion.items() if r > 0 and p < 0.05]
print("\nLectura: no es un efecto causal (el modelo no controla todo lo que podría estar detrás y el crédito puede ir a "
      "municipios que de entrada tienen menos presión sobre el bosque), sino una asociación estadística.")
if neg:
    print("  Más valor se asocia con MENOS riesgo en:", "; ".join(neg))
if pos:
    print("  Más valor se asocia con MÁS riesgo en:", "; ".join(pos))

### 7.4 Municipios de mayor riesgo (lista, sin necesitar el mapa)

La misma priorización territorial de 6.3, pero municipio por municipio y con la razón concreta: los
municipios-año de mayor riesgo predicho en 2025 (el año de prueba), cada uno con las variables que más empujaron
su predicción según SHAP, y la **principal palanca de política** (la variable de intervención —nunca bosque ni
área— con mayor aporte en esa fila). Como en el gradient boosting, el resultado se lee con honestidad: si la
palanca no varía de un municipio a otro, lo que hay no es "15 razones distintas" sino un arquetipo que se repite.

In [ ]:
N_LISTA = 15
idx_riesgo = np.argsort(-prob_test)[:N_LISTA]
nombres_politica_set = set(nombres_politica)

filas_lista = []
for pos in idx_riesgo:
    fila = test.iloc[pos]
    contribuciones = pd.Series(shap_clas[pos], index=X_test.columns).rename(index=NOMBRES)
    top_razones = contribuciones.sort_values(ascending=False).head(3)
    razon_texto = "; ".join(f"{nombre} ({valor:+.2f})" for nombre, valor in top_razones.items())
    solo_politica = contribuciones[contribuciones.index.isin(nombres_politica_set)]
    top_politica = solo_politica.sort_values(ascending=False).head(1)
    filas_lista.append({
        "municipio": fila.nombre_municipio, "departamento": fila.nombre_departamento,
        "riesgo predicho": prob_test[pos], f"¿deforestó en {ANIO_PRUEBA}?": "Sí" if fila.deforesta == 1 else "No",
        f"tasa real {ANIO_PRUEBA} (%)": fila.tasa,
        "top 3 SHAP (incluye estructurales)": razon_texto,
        "principal palanca de política (solo intervención)": f"{top_politica.index[0]} ({top_politica.iloc[0]:+.2f})",
    })

lista_riesgo = pd.DataFrame(filas_lista)
lista_riesgo.pipe(guardar_resultado, "random_forest_lista_municipios_mayor_riesgo", indice=False)
n_palancas = lista_riesgo["principal palanca de política (solo intervención)"].str.split(" (", regex=False).str[0].nunique()
print(f"De los {N_LISTA} municipios-año de mayor riesgo predicho en {ANIO_PRUEBA}, "
      f"{(lista_riesgo[f'¿deforestó en {ANIO_PRUEBA}?'] == 'Sí').sum()} sí deforestaron realmente ese año.")
print(f"Palancas de política distintas que aparecen en el top {N_LISTA}: {n_palancas}"
      + (" — un solo arquetipo de riesgo que se repite." if n_palancas == 1 else "."))
lista_riesgo

### 7.5 Importancia por bloques y por actividad productiva

Cuando dos variables están muy correlacionadas (crédito a café y área de café; crédito ganadero y bovinos), el
bosque se las reparte y la permutación de cada una por separado subestima a las dos. La solución es barajar
**juntas** todas las variables de un grupo y medir cuánto empeora el modelo. Se hace sobre 2025 (30 repeticiones,
caída del AUC para ¿deforesta?; caída del R² para ¿cuánto?), con los mismos grupos en los tres modelos.
Responde con una cifra a "¿cuánto aporta la ganadería, el café, el cacao...?", que es la lectura por cadena que pide ODM-2.

In [ ]:
imp_bloques = {"bloques": permutar_bloques(clasificador, X_test, y_test_clas, BLOQUES, puntaje_auc),
               "actividades": permutar_bloques(clasificador, X_test, y_test_clas, ACTIVIDADES, puntaje_auc)}
graficar_bloques(imp_bloques, "AUC")
print("¿deforesta? — por bloque:")
display(imp_bloques["bloques"].round(4))
print("¿deforesta? — por actividad productiva:")
display(imp_bloques["actividades"].round(4))

X_test_def = X_test[en_deforesta_test]
imp_bloques_reg = {"bloques": permutar_bloques(regresor, X_test_def, y_test_tasa_real, BLOQUES, puntaje_r2),
                   "actividades": permutar_bloques(regresor, X_test_def, y_test_tasa_real, ACTIVIDADES, puntaje_r2)}
print("Dado que deforesta, ¿cuánto? — por actividad productiva (caída del R²; muy ruidosa por Pedraza y por el poco R² de base):")
imp_bloques_reg["actividades"].round(4)

In [ ]:
act = imp_bloques["actividades"]
sig = act[(act.caida > 0) & (act.p_valor < 0.05)]
print("Actividades cuya permutación empeora el AUC de forma significativa (p < 0,05):",
      ", ".join(f"{i.split(' (')[0]} ({r.caida:.3f})" for i, r in sig.iterrows()) or "ninguna")
print(f"La actividad que más aporta es {act.index[0].split(' (')[0]} y la que menos, {act.index[-1].split(' (')[0]}. "
      f"Todas las actividades juntas (crédito, cultivos y ganado) pesan {imp_bloques['bloques'].drop('Estructura (bosque y área)').caida.sum():.3f} de AUC "
      f"frente a {imp_bloques['bloques'].loc['Estructura (bosque y área)', 'caida']:.3f} de lo estructural.")

**Lectura: ganadería y cacao, y la dirección del crédito (ODM-2, ODM-3).**

- **Por actividad, la ganadería es la que más aporta** (caída de AUC de 0,024 al barajar crédito ganadero, bovinos y fincas juntos), seguida de **cacao** (0,017) y café (0,008); la palma casi no aporta (0,0001, no significativa). Todas
  las actividades juntas pesan 0,046 de AUC frente a 0,331 de lo estructural: el crédito, los cultivos y el ganado, en conjunto, son una séptima parte de lo que pesan bosque y área.
- **Dirección del efecto.** Según SHAP, más crédito a café (rho = -0,66) y más crédito ganadero (rho = -0,44) se asocian con menos riesgo; más crédito a cacao, con más (rho = +0,87). Siguen siendo asociaciones: el crédito llega sobre todo a zonas de agricultura consolidada, donde ya hay menos presión sobre el bosque.
- **Para el objetivo:** ganadería y cacao son las dos cadenas que más aportan (permutación por actividad). Es lo que ODM-2 pide como cruce de financiamiento y producción, con la cautela de que
  la suma por actividad favorece a las que tienen más variables (ganadería suma 3, café y cacao 2).

### 7.6 Dependencia parcial

Cómo cambia la probabilidad media predicha al mover **una** variable de intervención, promediando sobre las demás
(las marcas del eje son los deciles de los datos). A diferencia de SHAP, no reparte nada entre variables: muestra
la forma de la relación aprendida. Sirve para detectar los umbrales y las relaciones no monótonas que el
coeficiente de una regresión no puede mostrar. Se muestran las seis variables de intervención de mayor
importancia por permutación.

In [ ]:
top_pdp = perm_clas[perm_clas.tipo == "intervención"].clave.head(6).tolist()
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, var in zip(axes.ravel(), top_pdp):
    pdp = partial_dependence(clasificador, X_train, [var], grid_resolution=30, kind="average", response_method="predict_proba")
    ax.plot(pdp["grid_values"][0], pdp["average"][0], color="#c8541a", lw=2)
    deciles = np.nanpercentile(X_train[var], np.arange(10, 100, 10))
    ax.plot(deciles, np.full(len(deciles), ax.get_ylim()[0]), "|", color="black", ms=12)
    ax.set(xlabel=NOMBRES[var], ylabel="prob. media de deforestar")
fig.suptitle("Dependencia parcial de las 6 variables de intervención más importantes (¿deforesta?)")
plt.tight_layout()
plt.show()

## 8. Guardar resultados

Las métricas y las predicciones quedan en tablas de `DATASET_RESULTADOS` (los modelos entrenados no se guardan: BigQuery guarda tablas, no archivos, y correr el notebook los reconstruye iguales), con **el mismo esquema** que
`gradient_boosting_metricas.csv` (columnas `modelo`, `métrica`, `prueba_temporal_2025`,
`prueba_departamentos_fuera`, `brecha_%`), para que `16_modelos_comparacion` los lea sin traducciones. Además se
guardan las predicciones de 2025 y las "limpias" de la prueba por departamentos, que permiten comparar los modelos
municipio por municipio (¿coinciden en a quién señalan?).

In [ ]:

def fila(modelo, metrica, temporal, deptos, brecha=np.nan):
    return {"modelo": modelo, "métrica": metrica, "prueba_temporal_2025": temporal,
            "prueba_departamentos_fuera": deptos, "brecha_%": brecha}


CLAS, REG = "Random forest — ¿deforesta?", "Random forest — ¿cuánto?"
metricas = pd.DataFrame([
    fila(CLAS, "exactitud balanceada", ba_test, ba_departamentos, brecha_clas),
    fila(CLAS, "AUC", m_clas_test["auc"], m_clas_oof["auc"]),
    fila(CLAS, "precisión", m_clas_test["precision"], m_clas_oof["precision"]),
    fila(CLAS, "recall / sensibilidad", m_clas_test["recall"], m_clas_oof["recall"]),
    fila(CLAS, "especificidad", m_clas_test["especificidad"], m_clas_oof["especificidad"]),
    fila(CLAS, "F1", m_clas_test["f1"], m_clas_oof["f1"]),
    fila(CLAS, "precisión promedio (AP, área bajo precisión-recall)", m_clas_test["ap"], m_clas_oof["ap"]),
    fila(REG, "R²", r2_test, r2_departamentos, brecha_reg),
    fila(REG, "RMSE", rmse_test, rmse_departamentos),
    fila(REG, "error absoluto promedio (MAE)", mae_test, mae_departamentos),
    fila(REG, "error absoluto mediano (MedAE)", medae_test, medae_departamentos),
    fila(REG, "R² en escala log (diagnóstico robusto)", m_reg_test["r2_log"], m_reg_oof["r2_log"]),
    fila(REG, "correlación de Spearman (diagnóstico robusto)", m_reg_test["spearman"], m_reg_oof["spearman"]),
    fila(REG, "R² excluyendo a Pedraza (diagnóstico, 5.3)", r2_sin_pedraza, np.nan, brecha_sin_pedraza),
    fila(REG, "R² una sola regresión sin separar, solo donde sí deforestó (diagnóstico, 5.2)",
         r2_score(y_test_tasa_real, pred_directo_def), np.nan),
])
metricas.pipe(guardar_resultado, "random_forest_metricas", indice=False)
metricas

pd.DataFrame({"clasificación": {k: str(v) for k, v in params_clas.items()},
              "regresión": {k: str(v) for k, v in params_reg.items()}}).pipe(guardar_resultado, "random_forest_hiperparametros", indice=True)
perm_clas.pipe(guardar_resultado, "random_forest_importancia_permutacion_clasificacion", indice=False)
perm_reg.pipe(guardar_resultado, "random_forest_importancia_permutacion_regresion", indice=False)
pd.concat({"bloques": imp_bloques["bloques"], "actividades": imp_bloques["actividades"]}).pipe(guardar_resultado, "random_forest_importancia_bloques", indice=True)
especificaciones_tabla.pipe(guardar_resultado, "random_forest_especificaciones", indice=True)
test.assign(prob_deforesta=prob_test, pred_deforesta=pred_test).assign(
    tasa_predicha=pd.Series(regresor.predict(X_test), index=test.index))[
    ["cod_dane", "nombre_municipio", "nombre_departamento", "anio", "deforesta", "tasa", "prob_deforesta", "pred_deforesta",
     "tasa_predicha"]].pipe(guardar_resultado, "random_forest_predicciones_prueba", indice=False)
m.assign(prob_deforesta=prob_clas_oof.values, pred_deforesta=pred_clas_oof.values, tasa_predicha=pred_tasa_oof.values)[
    ["cod_dane", "nombre_municipio", "nombre_departamento", "anio", "deforesta", "tasa", "prob_deforesta", "pred_deforesta",
     "tasa_predicha"]].pipe(guardar_resultado, "random_forest_predicciones_departamentos_fuera", indice=False)


## 9. Síntesis: qué responde este notebook de los objetivos del documento

Cada fila de la tabla de abajo se **calcula** con los resultados de este notebook (no está escrita a mano), de
modo que no puede contradecir a las secciones anteriores. Está pensada para usarse directamente en la sustentación
(coherencia entre presentación y documento) y para que las tres técnicas se lean con la misma plantilla.

Antes, tres lecturas de negocio que el gradient boosting todavía no hace y que responden de forma explícita a
OBJ-N1/ODM-2 y a CDM-6.

### 9.1 ¿Dónde coinciden alto crédito y alto riesgo? (OBJ-N1 y ODM-2)

Se miran dos formas de "alto crédito" porque cuentan historias distintas: el **volumen** (cuánta plata llega al municipio) y la **intensidad** (cuánta por km²). Los municipios grandes de la frontera amazónica pueden recibir mucho en volumen y casi nada por km².

In [ ]:
# OBJ-N1 exige identificar "al menos un municipio o cadena donde coincidan alto financiamiento agropecuario
# y alto riesgo de deforestacion" (ODM-2). Con las probabilidades "limpias" de la seccion 6 (cada municipio
# predicho por un modelo que nunca vio su departamento) se promedia el riesgo de cada municipio en 2021-2025 y
# se cruza con su financiamiento de dos maneras, que cuentan cosas distintas:
#   - VOLUMEN: credito de produccion acumulado 2021-2025 (millones de pesos). Responde "¿cuanta plata llega?".
#   - INTENSIDAD: credito por km² (la forma en que lo usan los modelos). Responde "¿que tan densa es?".
CADENAS_CRED = {"credito_ganaderia_produccion_mm": "Ganadería", "credito_cafe_produccion_mm": "Café",
                "credito_cacao_produccion_mm": "Cacao", "credito_palma_produccion_mm": "Palma",
                "credito_arroz_produccion_mm": "Arroz"}
# Las columnas de texto se pasan a str: si vienen como `category`, groupby arma TODAS las combinaciones posibles
# (producto cartesiano, millones de "municipios" vacios) y los cortes por cuartiles quedan en cero.
cred_abs = anual[["cod_dane", "anio", "credito_produccion_mm"] + list(CADENAS_CRED)].fillna(0).astype({"cod_dane": str})
por_mun = (m[["cod_dane", "anio", "nombre_municipio", "nombre_departamento", "tasa", "def_ideam_ha", "cred"]]
           .astype({"cod_dane": str, "nombre_municipio": str, "nombre_departamento": str})
           .assign(riesgo=prob_clas_oof.values).merge(cred_abs, on=["cod_dane", "anio"], how="left")
           .groupby(["cod_dane", "nombre_municipio", "nombre_departamento"], observed=True)
           .agg(riesgo_medio=("riesgo", "mean"), ha_total=("def_ideam_ha", "sum"), credito_total_mm=("credito_produccion_mm", "sum"),
                credito_por_km2=("cred", "mean"), **{c: (c, "sum") for c in CADENAS_CRED}).reset_index())
assert len(por_mun) <= m.cod_dane.nunique(), "por_mun debe tener un renglon por municipio"
por_mun["cadena_dominante"] = por_mun[list(CADENAS_CRED)].idxmax(axis=1).map(CADENAS_CRED)
corte_riesgo = por_mun.riesgo_medio.quantile(0.75)
por_mun["alto_riesgo"] = por_mun.riesgo_medio >= corte_riesgo
resumen = {}
for nombre, col in [("volumen", "credito_total_mm"), ("intensidad", "credito_por_km2")]:
    corte = por_mun[col].quantile(0.75)
    por_mun[f"alto_credito_{nombre}"] = por_mun[col] >= corte
    sel = por_mun[por_mun.alto_riesgo & por_mun[f"alto_credito_{nombre}"]]
    resumen[nombre] = dict(corte=corte, n=len(sel), pct_ha=100 * sel.ha_total.sum() / por_mun.ha_total.sum())
    print(f"{nombre.upper()}: {len(sel)} de {len(por_mun)} municipios están en el cuarto superior de riesgo predicho "
          f"(≥ {corte_riesgo:.2f}) Y en el cuarto superior de crédito por {nombre} (≥ {corte:,.1f}); concentran el "
          f"{resumen[nombre]['pct_ha']:.1f} % de las hectáreas deforestadas del IDEAM {ANIO_INICIO}-{ANIO_PRUEBA}.")
coinciden = por_mun[por_mun.alto_riesgo & por_mun.alto_credito_volumen].sort_values("ha_total", ascending=False)
cruce = (por_mun.groupby("cadena_dominante", observed=True)
         .agg(municipios=("cod_dane", "size"), alto_riesgo=("alto_riesgo", "sum"),
              alto_riesgo_y_alto_volumen=("alto_credito_volumen", lambda s: int((s & por_mun.loc[s.index, "alto_riesgo"]).sum())),
              ha_deforestadas=("ha_total", "sum")).sort_values("ha_deforestadas", ascending=False))
print("\nPor cadena dominante del municipio (la de mayor crédito acumulado):")
display(cruce.round(0).astype(int))
print("Municipios con alto riesgo y alto volumen de crédito, por hectáreas deforestadas:")
coinciden[["nombre_municipio", "nombre_departamento", "riesgo_medio", "credito_total_mm", "credito_por_km2",
           "cadena_dominante", "ha_total"]].head(10).round(2)

**Lectura: qué responde el cruce alto crédito-alto riesgo (OBJ-N1 y ODM-2).**

- **La coincidencia es de volumen, no de intensidad.** 48 de 639 municipios están a la vez en el cuarto superior de riesgo y en el cuarto superior de crédito por volumen, y concentran el 54,9 % de las hectáreas deforestadas 2021-2025. Si el
  crédito se mide por km², no coincide ninguno. La plata llega en monto grande a los municipios grandes de la frontera, pero delgada por km².
- **La ganadería domina el cruce** (42 de los 48). Los municipios con más hectáreas son Cartagena del Chairá y San Vicente del Caguán (Caquetá), Mapiripán (palma), La Macarena y San José del Guaviare. Para OBJ-N1 es el municipio o la cadena
  que se pide identificar; para ODM-2, el foco son las condiciones del crédito de ganadería en el arco Caquetá-Meta-Guaviare, no su monto.
- **Es una coincidencia descriptiva:** no prueba que el crédito cause la deforestación ni que retirarlo la reduzca. Y como el riesgo está prácticamente saturado en 1,0 para los municipios más riesgosos, el cruce no permite ordenar entre ellos.

### 9.2 Reproducibilidad (CDM-6)

Una prueba explícita en lugar de una afirmación: se vuelve a entrenar el clasificador con los mismos
hiperparámetros y la misma semilla, pero con un solo núcleo en vez de todos, y se comprueba que el bosque es el
mismo (mismos árboles, misma clasificación de los 639 municipios; las probabilidades coinciden hasta el error de
redondeo de sumar 500 árboles en otro orden). La búsqueda
usa semilla fija en cada bosque, así que el resultado tampoco depende de cuántos núcleos tenga la máquina.

In [ ]:
repetido = RandomForestClassifier(**FIJOS_RF, **params_clas, n_jobs=1).fit(X_train, y_train_clas)
prob_rep = repetido.predict_proba(X_test)[:, 1]
mismos_arboles = all(a.tree_.node_count == b.tree_.node_count
                     for a, b in zip(clasificador.estimators_, repetido.estimators_))
dif_max = float(np.abs(prob_rep - prob_test).max())
misma_clasificacion = bool(np.array_equal(prob_rep >= UMBRAL, prob_test >= UMBRAL))
reproducible = mismos_arboles and misma_clasificacion and dif_max < 1e-9
print(f"Reentrenar con la misma semilla (SEMILLA = {SEMILLA}) y un solo núcleo, en lugar de todos: "
      f"mismos árboles: {'sí' if mismos_arboles else 'no'}  ·  diferencia máxima de probabilidad: {dif_max:.1e}  ·  "
      f"misma clasificación de los 639 municipios: {'sí' if misma_clasificacion else 'no'}.")
print("(Una diferencia de ~1e-16 es solo el orden en que se suman las probabilidades de los 500 árboles según el número de núcleos.)")

### 9.3 Tabla de síntesis por objetivo

In [ ]:
cred_robustas = perm_clas[perm_clas.clave.isin(CREDITO) & (perm_clas.p_holm < 0.05)].clave.tolist()
_sentido = lambda v: ("menos riesgo cuando sube" if direccion[v][0] < 0 else "más riesgo cuando sube") if v in direccion else "sentido en 7.3"
if cred_robustas:
    texto_credito_rf = ("Líneas de crédito con asociación robusta (bootstrap por departamento y Holm): "
                        + "; ".join(f"{NOMBRES[v]} ({_sentido(v)})" for v in cred_robustas) + ". ")
else:
    texto_credito_rf = "Ninguna línea de crédito resiste el bootstrap por departamento con Holm. "
if imp_clas_credito.index[0] not in [NOMBRES[v] for v in cred_robustas]:
    texto_credito_rf += (f"Por impureza (MDI) la línea de mayor peso es {imp_clas_credito.index[0]}, pero esa lectura no trae incertidumbre y "
                         f"su permutación no resiste el bootstrap: no debe presentarse como hallazgo.")
a_est, b_est = especificaciones_tabla.loc["A. Estructural", "AUC"], especificaciones_tabla.loc["B. Principal (15 variables)", "AUC"]
cob = cobertura_oof[30]
dep_top = por_depto.index[:3].tolist()
estado_brecha = lambda b: "cumple" if abs(b) <= 15 else "no cumple"
reg_brecha_txt = (f"R² {r2_train:.2f} → {r2_test:.2f} (brecha {brecha_reg:.0f} %, {estado_brecha(brecha_reg)}); "
                  f"sin Pedraza {r2_sin_pedraza:.2f} (brecha {brecha_sin_pedraza:.0f} %, {estado_brecha(brecha_sin_pedraza)})")

sintesis = pd.DataFrame([
    {"objetivo / criterio": "OBJ-N1 · priorización territorial",
     "evidencia en este notebook": (f"AUC {auc_test:.3f} en {ANIO_PRUEBA} y {m_clas_oof['auc']:.3f} con departamentos fuera; revisando el 30 % de "
                                    f"municipios-año de mayor riesgo se cubre el {cob['pct_municipios']:.0f} % de los que deforestan y el "
                                    f"{cob['pct_ha']:.0f} % de las hectáreas. Departamentos de mayor riesgo: {', '.join(dep_top)}. "
                                    f"Alto riesgo y alto volumen de crédito coinciden en {resumen['volumen']['n']} municipios ({resumen['volumen']['pct_ha']:.0f} % de las hectáreas), "
                                    f"pero solo en {resumen['intensidad']['n']} si el crédito se mide por km² (9.1)."),
     "estado": "evidencia técnica aportada"},
    {"objetivo / criterio": "CEN-2 · validación por actores externos",
     "evidencia en este notebook": "No se mide aquí: requiere la validación de actores del mercado objetivo. Este notebook aporta la cifra verificable (curva de ganancia, ranking) que se les presenta.",
     "estado": "no aplica en el notebook"},
    {"objetivo / criterio": "OBJ-N2 / CEN-1 · autosostenibilidad",
     "evidencia en este notebook": "Ningún modelo cierra una venta por sí mismo; aporta el activo reproducible (notebook, modelo, hallazgos) que el Observatorio puede mostrar.",
     "estado": "no aplica en el notebook"},
    {"objetivo / criterio": "OBJ-N3 / CEN-3 / CDM-6 · activo reutilizable y reproducible",
     "evidencia en este notebook": f"Semilla fija y resultados guardados en BigQuery; reentrenar con otro número de núcleos da el mismo bosque y la misma clasificación: {'sí' if reproducible else 'no'}.",
     "estado": "cumple" if reproducible else "revisar"},
    {"objetivo / criterio": "ODM-2 · cruce de financiamiento y producción",
     "evidencia en este notebook": (f"Por actividad (permutación del AUC), la que más aporta es {act.index[0].split(' (')[0]} y la que menos, "
                                    f"{act.index[-1].split(' (')[0]}. " + texto_credito_rf),
     "estado": "evidencia técnica aportada"},
    {"objetivo / criterio": "ODM-3 / CDM-5 · variables con asociación significativa",
     "evidencia en este notebook": (f"¿deforesta?: de 13 variables de intervención, {cdm5_clas['interv_sig_gb']} con p < 0,05 (prueba t, como el gradient boosting), "
                                    f"{cdm5_clas['interv_sig_territorio']} con bootstrap por departamento y {cdm5_clas['interv_sig_holm']} tras Holm "
                                    f"({', '.join(cdm5_clas['nombres']) or 'ninguna'}). ¿cuánto?: {cdm5_reg['interv_sig_holm']} tras Holm. "
                                    f"La inferencia con errores robustos por conglomerado es la de la regresión logística."),
     "estado": "cumple (≥ 3)" if cdm5_clas["interv_sig_holm"] >= 3 else "no cumple (< 3)"},
    {"objetivo / criterio": "CDM-7 · brecha entrenamiento-prueba ≤ 15 %",
     "evidencia en este notebook": (f"¿deforesta?: dentro de la muestra {brecha_clas:.0f} %, out-of-bag {brecha_clas_oob:.0f} %, "
                                    f"validación por departamentos {brecha_clas_cv:.0f} %. ¿cuánto?: {reg_brecha_txt}."),
     "estado": ("cumple en ¿deforesta?; en ¿cuánto? no cumple" + (" (cumple sin Pedraza)" if abs(brecha_sin_pedraza) <= 15 else "")
                if abs(brecha_clas_oob) <= 15 and abs(brecha_clas_cv) <= 15 else "revisar")},
    {"objetivo / criterio": "REQ-04 · modelos interpretables, sin cajas negras",
     "evidencia en este notebook": "Importancia por impureza y por permutación (con valor p), por bloques y por actividad; SHAP y dependencia parcial para la dirección; lista de municipios con la razón de cada uno.",
     "estado": "cumple"},
    {"objetivo / criterio": "Lectura global para la sustentación",
     "evidencia en este notebook": (f"(1) Priorizar sí se puede: AUC {auc_test:.2f} en {ANIO_PRUEBA} y {m_clas_oof['auc']:.2f} en departamentos nuevos. "
                                    f"(2) Casi todo lo explica la estructura: bosque y área solos dan AUC {a_est:.3f} y con las 15 variables {b_est:.3f} "
                                    f"({b_est - a_est:+.3f}). (3) La magnitud es débil en valor (R² {r2_test:.2f}) pero el modelo ordena razonablemente "
                                    f"(Spearman {m_reg_test['spearman']:.2f}). (4) El alto crédito coincide con el alto riesgo por volumen "
                                    f"({resumen['volumen']['n']} municipios), no por intensidad ({resumen['intensidad']['n']})."),
     "estado": "resumen"},
])
sintesis.pipe(guardar_resultado, "random_forest_sintesis_objetivos", indice=False)
pd.set_option("display.max_colwidth", 400)
sintesis

## Próximos pasos y límites

- **Los tres modelos hablan el mismo idioma.** Las tres técnicas comparten muestra, variables, partición, pliegues, umbral, regla de búsqueda (grilla exhaustiva con opción de ponderar clases) y métricas, y guardan sus resultados con el mismo
  esquema. La comparación está en `16_modelos_comparacion`: este notebook no decide cuál modelo es mejor.
- **Lo que solo tiene este notebook.** El *out-of-bag* y la dependencia parcial (SHAP y la lista de municipios los comparte el gradient boosting). La permutación con bootstrap por departamento y Holm, la importancia por bloques y las especificaciones A-D ya las comparte el gradient
  boosting. La inferencia con errores robustos y odds ratios es la de la regresión logística.
- **Qué sí soporta este modelo.** Priorizar territorios (AUC 0,930 en 2025 y 0,905 en departamentos nuevos, con brecha dentro de CDM-7), ordenar municipios por severidad esperada (Spearman 0,73) y señalar tres variables de intervención que resisten una
  prueba exigente en ¿deforesta?: fincas ganaderas, cacao y crédito ganadero.
- **Qué no soporta.** Cuantificar hectáreas (R² de 0,06, o 0,37 sin Pedraza); ordenar entre los municipios más riesgosos (la probabilidad se satura en 1,0); atribuir la magnitud a una variable (0 tras Holm en ¿cuánto?); ni afirmar causalidad o que el
  crédito "protege" el bosque.
- **La ventaja de 2025 no es la de territorios nuevos.** Con departamentos fuera el desempeño baja (AUC 0,905, especificidad 0,796), y es la cifra que corresponde al uso real; debe leerse junto con la de los otros modelos.
- **La magnitud sigue siendo el punto débil de las tres técnicas.** Pedraza y unos pocos municipios del corredor seco del Caribe concentran el error. Una variable que distinga los municipios de bosque casi nulo (por ejemplo `bosque_ideam_ha` sin dividir por el área)
  sería el siguiente paso.
- **Asociación, no causalidad.** Ni la importancia ni SHAP ni la dependencia parcial son efectos causales. Que el crédito de una cadena aparezca asociado a menos riesgo puede deberse a que se otorga donde ya hay menos presión sobre el bosque.
- **Panel mensual.** Sigue pendiente, igual que en el gradient boosting.
- **Reproducibilidad.** Semilla fija; reentrenar con un solo núcleo da las mismas probabilidades (diferencia de 3e-16) y la misma clasificación. Las cifras de las lecturas escritas arriba son las de la corrida del 20/11/2026; si se vuelve a correr con
  otras versiones de las librerías, hay que verificarlas contra las salidas de cada celda.